# Configure

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Packages
from os.path import join
from pathlib import Path
import yaml
from yaml.loader import SafeLoader
import geopandas as gpd
from shapely.geometry import Polygon
import numpy as np
import pandas as pd
import rioxarray as rio
import xarray as xr
import contextily as cx

import unsafe.download as undown
import unsafe.files as unfile
import unsafe.unzip as ununzip
import unsafe.exp as unexp
import unsafe.ddfs as unddf
import unsafe.ensemble as unens

In [ ]:
# Name the fips, statefips, stateabbr, and nation that
# we are using for this analysis
# We pass these in as a list even though the framework currently
# processes a single county so that it can facilitate that
# expansion in the future
# TODO - could make sense to define these in the future
# in json or other formats instead of as input in code
fips_args = {
    'FIPS': ['42101'], 
    'STATEFIPS': ['42'],
    'STATEABBR': ['PA'],
    'NATION': ['US']
}
FIPS = fips_args['FIPS'][0]
NATION = fips_args['NATION'][0]

In [ ]:
# We need to pass in a config file that sets up
# constants and the structure for downlading data
# For the directory structure of our case study, 
# we use the following 
ABS_DIR = Path().absolute().parents[0]

CONFIG_FILEP = join(ABS_DIR, 'config', 'config.yaml')
# Open the config file and load
with open(CONFIG_FILEP) as f:
    CONFIG = yaml.load(f, Loader=SafeLoader)

# Wildcards for urls
URL_WILDCARDS = CONFIG['url_wildcards']

# Get the file extensions for api endpoints
API_EXT = CONFIG['api_ext']

# Get the CRS constants
NSI_CRS = CONFIG['nsi_crs']

# Dictionary of ref_names
REF_NAMES_DICT = CONFIG['ref_names']

# Dictionary of ref_id_names
REF_ID_NAMES_DICT = CONFIG['ref_id_names']

# Coefficient of variation
# for structure values
COEF_VARIATION = CONFIG['coef_var']

# First floor elevation dictionary
FFE_DICT = CONFIG['ffe_dict']

# Number of states of the world
N_SOW = CONFIG['sows']

# Data for flood depth grids
# Get hazard model variables
HAZ_FILEN = CONFIG['haz_filename']
# Get CRS for depth grids
HAZ_CRS = CONFIG['haz_crs']
# Ensemble members
HAZ_NENS = CONFIG['haz_nens']
# Number of columns for each depth grid
HAZ_NCOLS = CONFIG['haz_ncols']
# Num rows for each depth grid
HAZ_NROWS = CONFIG['haz_nrows']
# Lower left x coordinate
HAZ_XLL = CONFIG['haz_xll']
# Lower left y coordinate
HAZ_YLL = CONFIG['haz_yll']
# Cell resolution
HAZ_RES = CONFIG['haz_res']
# NODATA values
HAZ_NODATA = CONFIG['haz_nodata']

# Get the files we need downloaded
DOWNLOAD = pd.json_normalize(CONFIG['download'], sep='_').T

# We can also specify the filepath to the
# raw data directory
FR = join(ABS_DIR, "data", "raw")

# And external - where our hazard data should be
FE = join(FR, "external")

# Set up interim and results directories as well
# We already use "FR" for raw, we use "FO" 
# because you can also think of results
# as output
FI = join(ABS_DIR, "data", "interim")
FO = join(ABS_DIR, "data", "results")

# "Raw" data directories for exposure, vulnerability (vuln) and
# administrative reference files
EXP_DIR_R = join(FR, "exp")
VULN_DIR_R = join(FR, "vuln")
REF_DIR_R = join(FR, "ref")
# Haz is for depth grids
HAZ_DIR_R = join(FE, "haz")
# Pol is for NFHL
POL_DIR_R = join(FR, "pol")

# Unzip directory 
UNZIP_DIR = join(FR, "unzipped")

# We want to process unzipped data and move it
# to the interim directory where we keep
# processed data
# Get the filepaths for unzipped data
# We unzipped the depth grids (haz) and 
# ddfs (vuln) into the "external"/ subdirectory
HAZ_DIR_UZ = join(UNZIP_DIR, "external", "haz")
POL_DIR_UZ = join(UNZIP_DIR, "pol")
REF_DIR_UZ = join(UNZIP_DIR, "ref")
VULN_DIR_UZ = join(UNZIP_DIR, "external", "vuln")

# "Interim" data directories
EXP_DIR_I = join(FI, "exp")
VULN_DIR_I = join(FI, "vuln")
REF_DIR_I = join(FI, "ref")
# Haz is for depth grids
HAZ_DIR_I = join(FI, "haz")
# Pol is for NFHL
POL_DIR_I = join(FI, "pol")

# Download and unzip data

In [ ]:
wcard_dict = {x: fips_args[x[1:-1]][0] for x in URL_WILDCARDS}
undown.download_raw(DOWNLOAD, wcard_dict,
                    FR, API_EXT)


In [ ]:
ununzip.unzip_raw(FR, UNZIP_DIR)

# Prepare data for ensemble

The study domain corresponds to 12 digit USGS hydrological unit code (HUC) watershed 020402031008. We will spatially merge the NSI structures and Philadelphia data to this extent. We will restrict the other downloaded geospatial data to objects that intersect with this (e.g., Census Tracts that overlap). We may clip these for plotting purposes later.

## Study area boundary

In [ ]:
CLIP_SHP_FILEP = join(HAZ_DIR_UZ, 'RIFT_domain', 'domain_1.shp')
clip_geo = gpd.read_file(CLIP_SHP_FILEP)

## Process exposure

We will start by subsetting several datasets to the envelope of our clip polygon and then do the processing on those subsets.

### Get subsets of NSI and Philly data

In [ ]:
# Load in the NSI and Philly assessor, parcel, and footprint data
nsi_gdf = unexp.get_nsi_geo(FIPS, NSI_CRS, EXP_DIR_R)

assess_cols = ['assessment_date', 'basements', 'building_code',
               'building_code_description', 'building_code_description_new',
               'category_code', 'category_code_description', 'census tract',
               'exterior_condition', 'garage_type', 'general_construction',
               'interior_condition','location', 'market_value',
               'market_value_date', 'number_stories', 'owner_1',
               'parcel_number', 'sale_date', 'sale_price',
               'quality_grade', 'taxable_building', 'exempt_building',
               'total_area', 'total_livable_area',
               'topography', 'unit', 'year_built',
               'other_building', 'garage_type',
               'year_built_estimate', 'zoning']
assess = gpd.read_file(join(EXP_DIR_R, FIPS, 'assess.geojson'),
                       mask=clip_geo, columns=assess_cols)

parcel = gpd.read_file(join(EXP_DIR_R, FIPS, 'parcel.geojson'),
                       mask=clip_geo)
bld_fp = gpd.read_file(join(EXP_DIR_R, FIPS, 'bldfp.geojson'),
                       mask=clip_geo)

#### NSI subset

We'll follow existing UNSAFE functions to get our NSI dataset. We're going to include any residential structure in occupancy type RES1 and RES3

In [ ]:
# Set the values that we pass into the get_struct_subset function
occtype_list=['RES1-1SNB', 'RES1-2SNB', 'RES1-1SWB', 'RES1-2SWB',
              'RES1-SLNB', 'RES1-SLWB', 'RES1-3SNB', 'RES1-3SWB',
              'RES3A', 'RES3B', 'RES3C', 'RES3D', 'RES3E', 'RES3F']
sub_string = 'occtype.isin(@occtype_list)'
nsi_filt = unexp.get_struct_subset(nsi_gdf,
                                   filter=sub_string,
                                   occtype_list=occtype_list)

EXP_OUT_FILEP = join(EXP_DIR_I, FIPS, 'nsi_res.gpkg')
unfile.prepare_saving(EXP_OUT_FILEP)

# Clip to our boundary to reduce file size
nsi_clip_out = gpd.clip(nsi_filt, clip_geo.to_crs(nsi_filt.crs))

# Write file
nsi_clip_out.to_file(EXP_OUT_FILEP)

# Helpful summaries
print('Total NSI structures: {}'.format(len(nsi_gdf)))
print('Total NSI res structures: {}'.format(len(nsi_filt)))
print('Total NSI res structures in study area: {}'.format(len(nsi_clip_out)))

#### Philly data subsets

We want to use the assessment data to identify residential structures. Then we will subset the building footprints and parcels correspondingly. To match up records, we will link `assess['parcel_number']` to `parcel['BRT_ID']` to `bld_fp['PARCEL_ID_NUM']`.

Condos will require extra processing. From the Maps@Phila.gov email: “The buildings are matched via their centroid to the PWD Parcels for their parcelid, they could use the parcelid to connect to PWD Parcels, then use the BRT_ID field in the PWD Parcels to get to the OPA Tax Accounts.  This won’t be the cleanest solution for condos, but there’s no real system for handling those anywhere.  You can tell [redacted] she’s welcome to point out any mismatches she finds directly to me, I’ve worked with her before on other projects.” We describe the condo processing approach above the corresponding cell block.

We start by processing the assessor data. We use the `building_code_description` column to identify RES1 and RES3 mappings by sampling records and checking the properties in street view apps (Google and Philadelphia's own) and Philadelphia Properties/Atlas apps. Some building code descriptions appear to uniformly map to RES1 or RES3, but some are mixed. For example, some buildings are coded as twin row homes, which we consider RES3, but their neighbor was demolished so effectively the property is RES1. For our 'main' sample, we use building footprint processing (to identify detached footprints) and sq. ft. statistics on individual row homes to identify likely RES1. For sensitivity checks, we use majority mappings for building code descriptions to occupancy type (and a few other checks). 

Below we split the `building_code_description` column in a way that gives us reduced form information for a subset of structure types we can look through manually. 

In [ ]:
def split_bld_code(bld_desc):

    """
    Split a building code description into tokens based on the first numeric value.

    This function takes a building code description and splits it into tokens where
    all text before the first numeric value becomes one token, and all subsequent
    words (including numeric values) become individual tokens.

    Parameters
    ----------
    bld_desc : str
        A string containing the building code description.
        Example: 'APT 2-4 UNITS 3.5 STY MAS'

    Returns
    -------
    list
        A list where the first element is all text before the first number (as one string),
        followed by all remaining words as individual elements.
        Example: ['APT', '2-4', 'UNITS', '3.5', 'STY', 'MAS']
        If no numeric values are found, returns the entire description as a single element list.

    Examples
    --------
    >>> split_bld_code('APT 2-4 UNITS 3.5 STY MAS')
    ['APT', '2-4', 'UNITS', '3.5', 'STY', 'MAS']
    
    >>> split_bld_code('DET W/GAR 2 STY MASONRY')
    ['DET W/GAR', '2', 'STY', 'MASONRY']
    """

    if bld_desc is None:
        return bld_desc

    # Split the building code description into words
    full_code = bld_desc.split()

    # Find the index of the first string with a number as first character
    first_num_idx = next((i for i, word in enumerate(full_code) if word[0].isdigit()), None)
    
    if first_num_idx is not None:
        # Join everything before the first number as one token
        prefix = ' '.join(full_code[:first_num_idx])
        # Keep remaining words as separate tokens
        remaining = full_code[first_num_idx:]
        return [prefix] + remaining
    else:
        return [' '.join(full_code)]

In [ ]:
print('Total tax records in study area: {}'.format(len(assess)))

# We want to retain structures with a building code description
assess_sub = assess[assess['building_code_description'].notnull()].copy()
# split up the building code description field
assess_sub.loc[:, 'bld_code_split'] = assess_sub['building_code_description'].apply(split_bld_code)

# get the occupancy type code and the remaining token 
# into separate columns
assess_sub.loc[:, 'bld_type'] = assess_sub['bld_code_split'].apply(lambda x: x[0])
assess_sub.loc[:, 'bld_code_rest'] = assess_sub['bld_code_split'].apply(lambda x: x[1:])
# helpful to have the rest as a single string for some inspections
# can drop the last token though (usually foundation type)
assess_sub.loc[:, 'bld_code_rest_str'] = assess_sub['bld_code_rest'].apply(lambda x: ' '.join(x[:-1]))

# We want to subset to the category codes that may have res buildings
cat_codes = ['1', '2', '3', '14']
assess_sub = assess_sub[assess_sub['category_code'].str.strip().isin(cat_codes)]

# We do not want "VACANT" 
assess_sub = assess_sub[~assess_sub['bld_type'].str.contains('VACANT')]

# We can also drop anything with empty bld_code_rest_str
assess_non_res = assess_sub[assess_sub['bld_code_rest_str'] == '']
assess_sub = assess_sub[assess_sub['bld_code_rest_str'] != '']

print('Sample of tax records in study area: {}'.format(len(assess_sub)))

Below, we take the reduced form building codes to sample 10 properties (or the number of properties in the new code) for manual checking. We generated two of these files to allow for two analysts to check each others mappings and converge on processing rules for main and sensitivity analyses. We comment out the sample writing lines to avoid overwriting data generated in our analysis. The files we generated and coded are available for others to inspect. They may also generate new samples (change the file suffix). 

In [ ]:
# sample a few records from each bld_type group
samples = assess_sub.groupby('bld_type').apply(lambda x: x.sample(n=min(10, len(x)))).reset_index(drop=True)
# write out the parcel numbers and a few other columns and start 
# checking the ddf pairing
check_cols = ['parcel_number', 'bld_type', 'bld_code_rest_str',
              'building_code', 'category_code_description', 'zoning']
# check_dir = join(EXP_DIR_I, 'check_records')
# file_suf = '020425.csv'
# check_filep = join(check_dir, 'check_codes_' + file_suf)
# unfile.prepare_saving(check_filep)
# samples[check_cols].to_csv(check_filep, index=False)

We don't want to use the parcel centroid as a way to link with the flood hazard. We want to use the building footprint. We have to link the assessor records to parcels and building footprints. We need the parcels dataset because that's how we can merge the building footprints in. 

First, we will drop the `bld_type` that we identified as not having any residential structures. The remaining records are our residential subset. 

In [ ]:
# Identified manually by evaluating partial (but sometimes full) samples of unique bld_type
drop_bld_codes = ['HOTEL', 'PRIV GAR']
assess_res = assess_sub[~assess_sub['bld_type'].isin(drop_bld_codes)].copy()

print('Sample of res tax records in study area: {}'.format(len(assess_res)))

We also want to add the taxable and exempt building value for our structure value

In [ ]:
assess_res['val_struct'] = assess_res['taxable_building'] + assess_res['exempt_building']

We should also subset based on acceptable exterior and interior condition
The [documentation](https://metadata.phila.gov/#home/datasetdetails/5543865f20583086178c4ee5/representationdetails/55d624fdad35c7e854cb21a4/?view_287_per_page=100&view_287_page=1) tells us for exterior condition:

7. VACANT – No occupancy. FHA, VA, FNMA signs may be on the property. Property has been secured with fresh plywood over doors and windows.
8. SEALED – Doors and windows have been covered over by plywood, tin, concrete block or stucco. No interior access.
9. STRUCTURALLY COMPROMISED, OPEN TO THE WEATHER - Some or no windows, no door or door open, evidence of past abuse by vandals such as graffiti, missing railings, deteriorated wood and metal, etc. Scorch marks and/or fire and water damage to exterior brick, siding, bays, etc. Broken windows with blackened and charred interior.

For interior: 

6. Vacant – No occupancy. FHA, VA, FNMA signs may be on the property.
Property has been secured with fresh plywood over doors and windows.
7. Sealed / Structurally Compromised, Open to the Weather –
Doors and windows have been covered over by plywood, tin, concrete block or
stucco. No interior access. Some or no windows, no door or door open, evidence
of past abuse by vandals such as graffiti, missing railings, deteriorated wood and
metal, etc. Scorch marks and/or fire and water damage to exterior brick, siding,
bays, etc. Broken windows with blackened and charred interior.


In [ ]:
assess_res = assess_res[(~assess_res['interior_condition'].str.strip().isin(['6', '7'])) &
                        (~assess_res['exterior_condition'].str.strip().isin(['7', '8', '9']))].copy()
print('Sample of occupied res tax records in study area: {}'.format(len(assess_res)))
print('Unique res addresses in study area: {}'.format(len(assess_res['location'].unique())))

These are helpful numbers to keep in mind. While they may seem like lower and upper bounds, I don't think the are. We know there are tax records that correspond to the same structure, so we expect to have less than 103995. But we also have records that correspond to multiple structures (e.g., apartment complexes), so that will tend to drive the final sample up. However, there are also tax records that have erroneous parcel numbers (do not link to a BRT_ID in the parcel data) and/or tax records that correspond to buildings that are no longer there (or sometimes new buildings that don't have a building footprint ID yet). 

These are benchmark numbers. If we come in around 100k, I think our processing did a good job. We should of course see what tax records are not represented in our final dataset and quantify *why* that's the case, but if we come in around 100k I think it's a good benchmark. Btw, NSI comes in at around 96k, so similar benchmark.  From 104k, data error in one of tax/parcle/bld_fp can drive the number down, as well as aggregation of records to single structures. But I am expecting a decent number of disaggregating a record to many building footprints. From my checks so far, I expect more aggregation than disaggregation (e.g., there are some condos with a hundred units but I haven't seen a single apartment complex with 100 buildings) so I could see us being closer to 95k than 104k, especially considering the data error as well. 

Most of the assessment records get matched to building footprints successfully but there are a few inconsistencies because of the parcel boundaries not overlapping with the building enough for the centroid method to work. There are a number of records matched to multiple footprints (the entire footprint) in a way that makes it ambiguous to know the structure footprint for the record. This is an experimental section to see if we can get the residential structure for each record better than we can using the existing linkages. This will include some assumption-driven processing about how to filter for garages and other appurtenant structures that will have analogues in more of a post-processing step for the existing linked data. Will probably treat this as an experimental notebook that demonstrates the results of the comparisons since for clarity in the main analysis we'll want to have the assumptions baked in for more readability.   

In [ ]:
# Start by overlaying the bld_fp with parcels
# Most records have direct links to assess_res through parcel_number/BRT_ID
# These are the ones we want to overlay - we will do links for
# nonmatched afterwards 

par_cols = ['BRT_ID', 'PARCEL_ID', 'ADDRESS', 'geometry']
tax_cols = ['parcel_number', 'bld_type', 'bld_code_rest',
            'building_code_description_new',
            'val_struct', 'number_stories',
            'basements', 'unit']

assess_res['has_parcel_match'] = assess_res['parcel_number'].isin(parcel['BRT_ID']).copy()

assess_linked = assess_res[assess_res['has_parcel_match']].copy()
assess_no_link = assess_res[~assess_res['has_parcel_match']].copy()

direct_matches = parcel[par_cols].merge(
    assess_linked[tax_cols],
    right_on='parcel_number',
    left_on='BRT_ID'
)

print('Direct tax-parcel matches: {}'.format(len(direct_matches)))

# We can also create indirect matches by limiting parcels
# to those not in direct_matches (based on BRT_ID)
# and then doing a spatial join with assess_no_link
# we want to only keep the first of entries with
# duplicate ids
parcel_no_match = parcel[~parcel['BRT_ID'].isin(direct_matches['BRT_ID'])]
indirect_matches = gpd.sjoin(parcel_no_match[par_cols],
                             assess_no_link[tax_cols + ['geometry']],
                             predicate='contains',
                             how='inner')
print('Tax-parcel matches from sp joins: {}'.format(len(indirect_matches)))
indirect_matches['geometry'] = indirect_matches['geometry'].normalize()
indirect_matches = indirect_matches.drop_duplicates(subset='geometry', keep='first')
print('"unique" tax-parcel matches post drop duplicates: {}'.format(len(indirect_matches)))

tax_pc_matches = pd.concat([direct_matches, indirect_matches], axis=0)

# tax records are uniquely linked to parcels unless
# they refer to condos/apts, in which case we only want
# to bring the remainder of those tax records in later for aggregating
# things like structure value and then dividing across
# building footprints on the parcel
# in cases where this is only one building footprint, 
# you'd just keep what you aggregated
pc_res_dir = gpd.GeoDataFrame(tax_pc_matches,
                              geometry=tax_pc_matches['geometry'],
                              crs=parcel.crs)

# Because we do an overlay with bld_fp, there are touching buildings
# treated as different bld_fp_o even though they overlap with the 
# same parcel. We want to get the unary union of these touching
# building footprints because for our purposes the
# spatial precision comes from any unique built structures
# located at a specific parcel
bld_fp_diss = bld_fp.dissolve().explode()
bld_fp_o = gpd.overlay(pc_res_dir, bld_fp_diss[['geometry']], how='intersection')

# Convert the new footprints to epsg 5070 for area calculations
bld_fp_o['m2_bld'] = bld_fp_o.to_crs(epsg='5070').area

# Drop links where area threshold not met
bld_fp_min_m2 = 10
bld_fp_drop = bld_fp_o.loc[bld_fp_o['m2_bld'] <= bld_fp_min_m2]
bld_fp_o = bld_fp_o.loc[bld_fp_o['m2_bld'] > bld_fp_min_m2]

# Bring back links where area threshold was not met
# but it's the only building reference available for the
# parcel
merge_back = pc_res_dir[~pc_res_dir['parcel_number'].isin(bld_fp_o['parcel_number'])]['parcel_number']
merge_back_bld = bld_fp_drop[bld_fp_drop['parcel_number'].isin(merge_back)]
bld_fp_o = pd.concat([bld_fp_o, merge_back_bld], axis=0)

# print out the number of unmatched parcels
unmatched = len(pc_res_dir[~pc_res_dir['parcel_number'].isin(bld_fp_o['parcel_number'])])
print('Unmatched tax-bld_fp in study area: {}'.format(unmatched))
matched = len(pc_res_dir[pc_res_dir['parcel_number'].isin(bld_fp_o['parcel_number'])])
print('Matched tax-bld_fp in study area: {}'.format(matched))
match_prop = (matched)/len(pc_res_dir)
print('Proportion of matched tax records in study area: {}'.format(match_prop))

In [ ]:
# Get a new id
# Records with identical geometry should have same building footprint id
# Because of direct_matches above, we will only have 1 record per
# group but this is a more generalizable solution than other options
bld_fp_o['geometry'] = bld_fp_o['geometry'].normalize()
bld_fp_o['bfid'] = bld_fp_o.groupby('geometry').ngroup()

# Calculate the number of parcels linked to this building footprint
bld_fp_o['n_parcels'] = bld_fp_o.groupby('bfid')['parcel_number'].transform('nunique')
# and vice versa
bld_fp_o['n_bld'] = bld_fp_o.groupby('parcel_number')['bfid'].transform('nunique')

Any building linked to one parcel gets assigned to that parcel.

For remaining cases, will require some combination of disaggregation of tax record to buildings and aggregating info from tax records before disaggregating to buildings. Note that some condos have 1:1 (i.e., no disaggregation required) and still need to be linked to unmatched tax records for aggregation. 

In [ ]:
# pc_bld will be our final dataframe of links
# we'll append processed subset dfs into a list
# and then concat into pc_bld when done
pc_bld_l = []

# Separate parcels with one building from more complex cases
# Add our simple cases to our processed dfs list
par_one_bld_match = bld_fp_o[bld_fp_o['n_bld'] == 1]
pc_bld_l.append(par_one_bld_match)
par_one_bld_many = bld_fp_o[bld_fp_o['n_bld'] > 1]

print('Number of 1 to 1 matches: {}'.format(len(par_one_bld_match)))
print('Number of 1 to many matches: {}'.format(len(par_one_bld_many['parcel_number'].unique())))

For the one parcel to many bld cases, we'll start by dropping any footprints that are a small proportion of the max building footprint associated with the parcel. We can sometimes see complexes with a building 2 to 3 times larger than others, maybe even a bit more, but it's very uncommon to have one structure much larger than the others. They tend to be a similar size anyway. So, we will drop footprints that are a small proportion. There will still be some cases that are ambiguous after this. We can split on bld_type for apt/condo vs. other structures because we can try to be a bit more restrictive with the area ratio for the latter and can do another round of checking. Also, we are more comfortable assuming the largest area structure is the main building for these, whereas for other apt/condo we are more comfortable assuming we need to disaggregate across the structures. 

In [ ]:
par_one_bld_many['area_ratio'] = (par_one_bld_many['m2_bld'] / 
                                  par_one_bld_many.groupby('parcel_number')['m2_bld'].transform('max'))

# This filter is arbitrary but QA checks suggests effective
# First, it has a rather high threshold for the different sizes of
# footprints linked to a parcel. This is a good filter to have for
# the structures we want to drop like garages, but there are some huge 
# complexes that have a lot of adjacent buildings or huge units
# and a few detached units that end up taking a small proportion. That's
# where the 100 m2 threshold comes in
pc_one_many_keep = par_one_bld_many.loc[(par_one_bld_many['area_ratio'] >= .95) |
                                        (par_one_bld_many['m2_bld'] >= 100)].copy()

# Calculate bld linked to each parcel
pc_one_many_keep['n_bld'] = pc_one_many_keep.groupby('parcel_number')['bfid'].transform('nunique')

# Separate parcels with one building from more complex cases
# Add our simple cases to our processed dfs list
pc_one_bld_lower_conf = pc_one_many_keep.loc[pc_one_many_keep['n_bld'] == 1]
pc_bld_l.append(pc_one_bld_lower_conf)
# Combo of pc to disagg and bld_type we'd like to 
# do a bit more processing on to filter out garages/similar
pc_one_bld_many = pc_one_many_keep.loc[pc_one_many_keep['n_bld'] > 1]

print('Lower confidence 1 to 1 matches: {}'.format(len(pc_one_bld_lower_conf)))
print('Remaining 1 to many matches: {}'.format(len(pc_one_bld_many['parcel_number'].unique())))
print('Remaining buildings: {}'.format(len(pc_one_bld_many)))

In [ ]:
# At this stage, we assume we need to disagg all APTS & RES CONDO
pc_disagg = pc_one_bld_many[pc_one_bld_many['bld_type'].isin(['APTS', 'RES CONDO'])].copy()
 
# For all else, we assume there is only supposed to be one structure
# We'll only keep structures if they are very similar in size to 
# the largest structure (.95 or higher proportion of area) 
pc_poss_garag = pc_one_bld_many[~pc_one_bld_many['bld_type'].isin(['APTS', 'RES CONDO'])].copy()
pc_wo_garag = pc_poss_garag[pc_poss_garag['area_ratio'] >= .95].copy()
pc_wo_garag['n_bld'] = pc_wo_garag.groupby('parcel_number')['bfid'].transform('nunique').copy()
# Separate parcels with one building from more complex cases
# Add our simple cases to our processed dfs list
pc_one_bld_lowest_conf = pc_wo_garag[pc_wo_garag['n_bld'] == 1]

# For those parcels where all buildings are less than 100 m2, we
# will just keep the largest (these look like misplaced parcels
# that catch half of two separate houses from all of our checks)
pc_wo_g_remain = pc_wo_garag[pc_wo_garag['n_bld'] > 1].sort_values('m2_bld', ascending=False)
pc_wo_g_one = pc_wo_g_remain[(pc_wo_g_remain['area_ratio'] == 1) &
                             (pc_wo_g_remain['m2_bld'] < 100)].drop_duplicates('parcel_number', keep='first')
pc_one_bld_lowest_conf = pd.concat([pc_one_bld_lowest_conf, pc_wo_g_one], axis=0)
pc_bld_l.append(pc_one_bld_lowest_conf)
pc_wo_g_disagg = pc_wo_g_remain[~pc_wo_g_remain['parcel_number'].isin(pc_wo_g_one['parcel_number'])]

# Add these few to our processed dfs list
pc_bld_main = pd.concat(pc_bld_l, axis=0)

# Add rest to pc_disagg
pc_disagg = pd.concat([pc_disagg, pc_wo_g_disagg], axis=0)

print('Lowest confidence 1 to 1 matches: {}'.format(len(pc_one_bld_lowest_conf)))
print('Remaining 1 to many matches: {}'.format(len(pc_disagg['parcel_number'].unique())))
print('Remaining buildings: {}'.format(len(pc_disagg)))
print('Overall 1 to 1 matches identified: {}'.format(len(pc_bld_main)))

We need to check if we are missing any tax records from `assess_linked` in our new datasets of tax records linked to one main building or tax records to disaggregate across structures. These can be missing because their building footprint is missing from `bld_fp`. Let's check out what's happening

In [ ]:
# These are records we can link to parcels but not to building footprints, 
# at least with the overlay
# dir_mat_missed = direct_matches[~(direct_matches['parcel_number'].isin(pc_bld_main['parcel_number'])) &
#                                 ~(direct_matches['parcel_number'].isin(pc_disagg['parcel_number']))]

# For example, this code will return an empty dataframe
# bld_fp_o[bld_fp_o['parcel_number'].isin(dir_mat_missed['parcel_number'])]

# But also can't find any of these records in the bld_fp data...
# dir_mat_missed[dir_mat_missed['PARCEL_ID'].isin(bld_fp['PARCEL_ID_NUM'])]
# dir_mat_missed[dir_mat_missed['ADDRESS'].isin(bld_fp['ADDRESS'])]

# Simply put, these are missing building footprints. See the following code for
# quick visualization of these instances
# Replace the BRT_ID with samples of BRT_ID from dir_mat_missed
# Some of these have structures but they're missing whereas others
# are vacant. I say we treat the Philly footprints as our baseline
# and treat these as examples of no building...
# We can use these parcel boundaries as a filter to remove NSI
# points inside of them as a sensitivity check

# from shapely.geometry import box
# import matplotlib.pyplot as plt
# temp = parcel[parcel['BRT_ID'] == '291124701']
# bbox = temp.total_bounds
# window = box(*bbox).buffer(.0001)

# fig, ax = plt.subplots()

# temp2 = bld_fp[bld_fp.geometry.intersects(window)]

# if not temp2.empty:
#     temp2.plot(ax=ax)
# temp.plot(ax=ax, color='none', edgecolor='red')


# Similarly, there are only 3 records below and they each seem to have an explanation for exclusion
# from further analysis. One is vacant according to recent satellite imagery. Another
# appears to have a missing building footprint in our data. Finally, one appears
# like it has incorrect links that even show up as problematic on the Properties web app

# indirect_matches[(~indirect_matches['parcel_number'].isin(pc_bld_main['parcel_number'])) &
#                   ~(indirect_matches['parcel_number'].isin(pc_disagg['parcel_number']))]

Now we want to aggregate tax records that represent buildings with many units and disaggregate tax records that represent parcels with many buildings. 

We check the parcels to disaggregate with the records we didn't link up to parcels. Some of these (maybe all) are the units in condos or apartment buildings and need to be aggregated with our parcels to disaggregate. If some of them don't link up, we have to check if we can link the tax record with one of our records in pc_bld_main, which suggests aggregating structure characteristics. Alternatively, we can see if we can link the tax record to a building footprint through a spatial join (through a parcel boundary and/or building footprint). Once we have no more stones unturned, we will have our set of records to disaggregate across structures. We also have to do aggregation in pc_bld_main for condos. 

In [ ]:
# Only need value for aggregation and parcel_number for groupby
agg_cols = ['val_struct', 'BRT_ID']

# To do aggregation, there are different steps
# we have to take for parcels in pc_bld_main or pc_disagg
# based on whether they had a direct match to parcel or not
# For those with a direct match, we can just directly link BRT_ID
# that records in assess_no_link will get with a gpd sjoin to parcel
pc_dir_match = parcel[parcel['BRT_ID'].isin(direct_matches['BRT_ID'])]
dir_match_sp = gpd.sjoin(assess_no_link,
                         pc_dir_match,
                         predicate='within')

# But for those without a BRT_ID link, if they have a counterpart
# for aggregation we have to find that out through a spatial join
# We can remake the indirect_matches gdf and then drop
# the records in pc_bld_main and pc_disagg that are inside that
# Then we can merge on BRT_ID like we could for direct matches
indirect_matches = gpd.sjoin(assess_no_link,
                             parcel_no_match,
                             predicate='within')
# Then we want to remove any records whose parcel_number is already in
# pc_bld_main or pc_disagg to avoid double counting
ind_mask = ((~indirect_matches['parcel_number'].isin(pc_bld_main['parcel_number'])) &
             ~indirect_matches['parcel_number'].isin(pc_disagg['parcel_number']))
ind_match_sp = indirect_matches.loc[ind_mask].copy()

# Now create a geodataframe of these two 
assess_sp_link = pd.concat([dir_match_sp, ind_match_sp], axis=0)

assess_sp_link = assess_sp_link.loc[:, agg_cols].copy()

# Find parcel matches in pc_bld_main for aggregation
pc_agg_match = assess_sp_link[assess_sp_link['BRT_ID'].isin(pc_bld_main['parcel_number'])]
# Same for pc_disagg
pc_disagg_match = assess_sp_link[assess_sp_link['BRT_ID'].isin(pc_disagg['parcel_number'])]

# Get corresponding records from each of pc_agg_match & pc_disagg_match 
# so we can do aggregation (and subsequent disagg where needed)
pc_agg_add = pc_bld_main[pc_bld_main['parcel_number'].isin(pc_agg_match['BRT_ID'])].copy()
# Add relevant agg characteristics to dataframe
pc_agg_add = pc_agg_add.loc[:, agg_cols].copy()
# Then concat them
pc_agg_proc = pd.concat([pc_agg_match, pc_agg_add], axis=0)

# Repeat for pc_disagg_match
pc_disagg_add = pc_disagg[pc_disagg['parcel_number'].isin(pc_disagg_match['BRT_ID'])].copy()
pc_disagg_add = pc_disagg_add.loc[:, agg_cols].copy()
pc_disagg_proc = pd.concat([pc_disagg_match, pc_disagg_add], axis=0)

# Aggregate structure values, make dict, replace vals in main df
pc_agg_sum = pc_agg_proc.groupby('BRT_ID', as_index=False)['val_struct'].sum()
pc_agg_dict = dict(zip(pc_agg_sum['BRT_ID'], pc_agg_sum['val_struct']))
a_mask = pc_bld_main['parcel_number'].isin(pc_agg_sum['BRT_ID'])
pc_bld_main.loc[a_mask, 'val_struct'] = pc_bld_main.loc[a_mask, 'parcel_number'].map(pc_agg_dict)

pc_disagg_sum = pc_disagg_proc.groupby('BRT_ID', as_index=False)['val_struct'].sum()
pc_disagg_dict = dict(zip(pc_disagg_sum['BRT_ID'], pc_disagg_sum['val_struct']))
d_mask = pc_disagg['parcel_number'].isin(pc_disagg_sum['BRT_ID'])
pc_disagg.loc[d_mask, 'val_struct'] = pc_disagg.loc[d_mask, 'parcel_number'].map(pc_disagg_dict)

Drop structures with no value. Several checks suggest these are demolished structures/vacant lots. Some checks don't suggest this, but it's such a small number we can remove in our "best guess" inventory. 

Next, merge other structure characteristics into pc_bld_main and pc_disagg (RES1 and RES3 best guesses come later). Disaggregate for parcels with many structures and merge into pc_bld_main (need a new dataframe for these records). 

Finally, we want to check what tax records were lost along the way in this processing. Can we directly do tax record in building footprint spatial joins to recover? After leaving no stone unturned, we will call our residential inventory final and write it out with a parsimonious set of columns. 

In [ ]:
inv_cols = ['parcel_number', 'basements', 'number_stories',
            'bld_type', 'bld_code_rest', 'val_struct', 
            'bfid', 'geometry']
assess_merge_cols = ['parcel_number', 'basements', 'number_stories',
                     'bld_type', 'bld_code_rest',
                     'building_code_description_new']

# Drop no val records
pc_bld_main_inv = pc_bld_main.loc[pc_bld_main['val_struct'] > 0].copy()
pc_disagg_inv = pc_disagg.loc[pc_disagg['val_struct'] > 0].copy()

no_val_main = len(pc_bld_main[~pc_bld_main['parcel_number'].isin(pc_bld_main_inv['parcel_number'])])
print('Dropped main records due to 0 value: {}'.format(no_val_main))
no_val_disagg = len(pc_disagg[~pc_disagg['parcel_number'].isin(pc_disagg_inv['parcel_number'])])
print('Dropped disagg records due to 0 value: {}'.format(no_val_disagg))

# Get ratio of area to sum to assign values
pc_disagg_inv['total_m2'] = pc_disagg_inv.groupby('parcel_number')['m2_bld'].transform('sum')
pc_disagg_inv['m2_ratio'] = pc_disagg_inv['m2_bld'] / pc_disagg_inv['total_m2']
pc_disagg_inv['val_struct'] = pc_disagg_inv['val_struct'] * pc_disagg_inv['m2_ratio']

pc_disagg_inv = pc_disagg_inv.drop(columns=['total_m2', 'm2_ratio'])

`pc_bld_main_inv` has every parcel with a single building linked to all of its appropriate records. `pc_disagg_inv` has a record for each unique building for one parcel to many building records. The value is distributed across each of these structures. Now we have to concat these datasets, generate a unique id for each structure inventory record, and finish the inventory with the remaining characteristics. 

We'll keep: bld_type, parcel_number, bfid, val_struct, basements, number_stories, stories_n and drop the rest

After that, we'll assign RES1 & RES3 based on bld_type mappings and adjacent building processing

In [ ]:
# Our main inventory
phil_inv = pd.concat([pc_bld_main_inv, pc_disagg_inv], axis=0)
phil_inv = phil_inv.drop(columns=['index_right', 'n_parcels',
                                  'n_bld', 'area_ratio', 'unit'])

# Number of stories (call stories_n)
apt_new_codes = ['APARTMENTS - BLT AS RESID',
                 'APTS - GARDEN', 'APARTMENTS - MID RISE', 'APTS - HIGH RISE']
non_apts_msk = ~((phil_inv['bld_type'].isin(['APT', 'APTS'])) |
                 (phil_inv['building_code_description_new'].isin(apt_new_codes)))
phil_inv.loc[non_apts_msk, 'stories_n'] = phil_inv.loc[non_apts_msk, 
                                                       'bld_code_rest'].apply(lambda x: x[0])

apt_msk = phil_inv['bld_type'] == 'APT'
phil_inv.loc[apt_msk, 'stories_n'] = phil_inv.loc[apt_msk,
                                                  'bld_code_rest'].apply(lambda x: x[2])

apts_fill = (((phil_inv['bld_type'] == 'APTS') | 
              (phil_inv['building_code_description_new'].isin(apt_new_codes)))
             & (phil_inv['number_stories'].notnull()))
phil_inv.loc[apts_fill, 'stories_n'] = phil_inv.loc[apts_fill, 'number_stories'].astype(int).astype(str).copy()

apts_st_miss = (phil_inv['bld_type'] == 'APTS') & (phil_inv['number_stories'].isnull())
phil_inv.loc[apts_st_miss, 'stories_n'] = '2+'

# Convert to 1 or 2 stories (for ddf purposes)
phil_inv.loc[phil_inv['stories_n'].isin(['1', '1.5']), 'ddf_stories'] = 1
phil_inv.loc[~phil_inv['stories_n'].isin(['1', '1.5']), 'ddf_stories'] = 2
                                                       
# Basement (b_type)
no_bsmt = phil_inv['basements'].isin(['0', '1', '2', '4']) | phil_inv['basements'].isnull()
phil_inv.loc[no_bsmt, 'b_type'] = 'NB'
phil_inv.loc[~no_bsmt, 'b_type'] = 'WB'

In [ ]:
# RES1 and RES3 mappings (occ_type)
res1_codes = ['DET', 'DET CONV APT',
              'DET OFF/STORE', 'DET OFF/STR', 'DET W/B GAR',
              'DET W/D GAR', 'DET W/GAR']
res1 = phil_inv['bld_type'].isin(res1_codes)
phil_inv.loc[res1, 'occ_type'] = 'RES1'

# Remainder are RES3
phil_inv.loc[~res1, 'occ_type'] = 'RES3'

# For those w/ res1, we should check if they touch another structure
# in which case they get a RES3 coding (twin houses, for instance)
res1_touch = gpd.sjoin(phil_inv.loc[res1],
                       bld_fp_o[['geometry']],
                       how='inner',
                       predicate='touches')
res1_touch_uniq = res1_touch['parcel_number']
phil_inv.loc[phil_inv['parcel_number'].isin(res1_touch_uniq),
             'occ_type'] = 'RES3'

# For those w/ res3, we should see which ones actually stand
# alone, particularly for the following types
res3_check = ['ROW', 'ROW B/GAR', 'ROW B/OFF-STR',
              'ROW CONV/APT', 'ROW W/DET GAR', 'ROW W/OFF STR',
              'S/D CONV APT', 'S/D OFF/STR', 'S/D W/B GAR', 'S/D W/D GAR',
              'S/D W/GAR', 'SEMI/DET', 'STR/OFF', 'STR/OFF+APT'] 

res3_det_gdf = gpd.sjoin(phil_inv.loc[phil_inv['bld_type'].isin(res3_check)],
                         bld_fp_o[['geometry']],
                         predicate='touches',
                         how='left')
res3_det = res3_det_gdf[res3_det_gdf['index_right'].isnull()]['parcel_number']
phil_inv.loc[phil_inv['parcel_number'].isin(res3_det),
             'occ_type'] = 'RES1'


In [ ]:
# We have some categories fully coded based on visual inspections, so should link those in here
# Some of the records won't be in phil_inv because they may have been dropped due to
# lacking a building footprint or poor interior/exterior condition. Many of these
# are units for aggregation in condos or apartments and you can find them in 
# indirect_matches or dir_match_sp. 
# Read in the check file and map parcel_number/occ_type mappings for any group
# that has less than 10 entries
# Can also check the codes assigned in phil_inv against our hand codings
# to see if our rules do a good job of capturing our visual inspection
hand_coded_otype_fp = join(EXP_DIR_I,
                           'check_records',
                           'check_codes_020625_consensus.csv')
hand_coded_occtypes = pd.read_csv(hand_coded_otype_fp,
                                  dtype={'parcel_number': 'str'})
hand_coded_occtypes = hand_coded_occtypes[hand_coded_occtypes['drop_final'] != 1]
hand_coded_occtypes['btype_count'] = hand_coded_occtypes.groupby('bld_type').transform('size')

hc_check = phil_inv[phil_inv['parcel_number'].isin(hand_coded_occtypes['parcel_number'])]
hc_check = hc_check.loc[:, ['parcel_number', 'occ_type']]

hc_comp = hc_check.merge(hand_coded_occtypes[['parcel_number', 'bld_type', 'ddf_type_final', 'btype_count']],
                         on='parcel_number')

hc_update = hc_comp[hc_comp['btype_count'] < 10]
hc_u_dict = dict(zip(hc_update['parcel_number'], hc_update['ddf_type_final']))

hc_sub = phil_inv['parcel_number'].isin(hc_update['parcel_number'])
phil_inv.loc[hc_sub, 'occ_type'] = phil_inv.loc[hc_sub, 'parcel_number'].map(hc_u_dict) 

# There are a few mismatches because of missing building footprint neighbors but
# in general this seems to lead to more RES1 than we hand coded, a conservative
# result because NSI has mostly RES1...
# hc_comp[hc_comp['occ_type'] != hc_comp['ddf_type_final']]


The above RES1/RES3 code probably results in more RES1 than visual one-by-one inspection would, but represents a conservative best guess. There are sensitivity checks we should run where we unifomrly assign RES1 or RES3 based on bld_type, which will result in far more RES3 than this set provides.

We are now ready to write out our best guess representation of the Philly residential structure inventory

In [ ]:
out_cols = ['parcel_number', 'val_struct', 'bld_type', 'stories_n',
            'bfid', 'ddf_stories', 'b_type', 'occ_type', 'geometry']
phil_inv_out = phil_inv.loc[:, out_cols].copy()
phil_out_fp = join(EXP_DIR_I, FIPS, 'phil_res.gpkg')
phil_inv_out.to_file(phil_out_fp)

## Process vulnerability

In [ ]:
unddf.process_naccs(VULN_DIR_UZ, VULN_DIR_I)

For this case study, we will create a three story with basement ddf for apartments that shifts the corresponding no basement ddf to the left by two in accordance with the min depth that HAZUS says apartments can experience damage.

In [ ]:
naccs_ddf = pd.read_parquet(join(VULN_DIR_I, 'physical', 'naccs_ddfs.pqt'))

# For 3SNB
temp = naccs_ddf[(naccs_ddf['ddf_type'] == '3SNB_RES3A')].copy()
max_params = temp.loc[temp['depth_ft'] == temp['depth_ft'].max(),
                      'params']
temp2 = temp.copy()
temp2['depth_ft'] = temp2['depth_ft'] - 2.0
temp['merge'] = np.round(temp['depth_ft']*100).astype(int)
temp2['merge'] = np.round(temp2['depth_ft']*100).astype(int)
temp2 = temp2.drop(columns='depth_ft')
temp3 = temp.merge(temp2, on=['merge'], how='left')

temp3['ddf_type'] = '3SWB_RES3A'
temp3['params'] = np.where(temp3['params_y'].notnull(),
                           temp3['params_y'],
                           max_params)
keep_cols = ['depth_ft', 'ddf_type', 'params']
naccs_res3b = temp3[keep_cols].copy()

# Repeat for 1SNB
temp = naccs_ddf[(naccs_ddf['ddf_type'] == '1SNB_RES3A')].copy()
max_params = temp.loc[temp['depth_ft'] == temp['depth_ft'].max(),
                      'params']
temp2 = temp.copy()
temp2['depth_ft'] = temp2['depth_ft'] - 2.0
temp['merge'] = np.round(temp['depth_ft']*100).astype(int)
temp2['merge'] = np.round(temp2['depth_ft']*100).astype(int)
temp2 = temp2.drop(columns='depth_ft')
temp3 = temp.merge(temp2, on=['merge'], how='left')

temp3['ddf_type'] = '1SWB_RES3A'
temp3['params'] = np.where(temp3['params_y'].notnull(),
                           temp3['params_y'],
                           max_params)
keep_cols = ['depth_ft', 'ddf_type', 'params']
naccs_res1b = temp3[keep_cols].copy()

naccs_out = pd.concat([naccs_ddf, naccs_res3b, naccs_res1b], axis=0)
naccs_out.to_parquet(join(VULN_DIR_I, 'physical', 'naccs_ddfs.pqt'))

In [ ]:
import json
naccs_max_filep = join(VULN_DIR_I, "physical", "naccs.json")
with open(naccs_max_filep, "r") as fp:
    naccs_max = json.load(fp)
naccs_max['3SWB_RES3A'] = naccs_max['3SNB_RES3A']
naccs_max['1SWB_RES3A'] = naccs_max['1SNB_RES3A']
with open(naccs_max_filep, "w") as fp:
        json.dump(naccs_max, fp)

## Process reference data

Clip reference data to our catchment area and link both NSI & Philly structures to ref data

In [ ]:
# Subset of ref downloads
ref_downloads = DOWNLOAD[DOWNLOAD.index.str.contains('_ref_')]
# Clip ref to catchment
# Use FIPS as clip_str since the catchment is in a county
# and this is eaiser for directory management
unexp.clip_ref_files(clip_geo, FIPS, fips_args, ref_downloads,
                     wcard_dict, REF_DIR_UZ, REF_DIR_I)

In [ ]:
# Link NSI to references
nsi_refs = unexp.get_ref_ids(nsi_clip_out.set_index('fd_id'), FIPS,
                             REF_ID_NAMES_DICT, REF_DIR_I, EXP_DIR_I)

nsi_ref_filep = join(EXP_DIR_I, FIPS, "nsi_ref.pqt")
unfile.prepare_saving(nsi_ref_filep)
nsi_refs.to_parquet(nsi_ref_filep)

# Link Philly to references
phil_refs = unexp.get_ref_ids(phil_inv_out.set_index('bfid'), FIPS,
                              REF_ID_NAMES_DICT, REF_DIR_I, EXP_DIR_I)
phil_ref_filep = join(EXP_DIR_I, FIPS, "phil_ref.pqt")
phil_refs.to_parquet(phil_ref_filep)

## Hazard

In [ ]:
# Get dict of filenames
# Ensemble number fo its filename
dg_fileps = {}

# Also need dict of depths
# Ensemble number to series
# Will turn into dataframes after the loop
# and write out as a .pqt file
nsi_depths = {}
phil_depths = {}

# Need projected crs for Phil centroid and depth id
phil_cent_reproj = phil_inv_out.to_crs(epsg=5070)
phil_cent_reproj['geometry'] = phil_cent_reproj['geometry'].centroid
# But want them to be in the same crs as our hazard data
phil_cent_reproj = phil_cent_reproj.to_crs(epsg=HAZ_CRS)
# Same with NSI data
nsi_clip_out = nsi_clip_out.to_crs(epsg=HAZ_CRS)

for i in range(1, HAZ_NENS + 1):
    # The filenames have a leading 0 before the ens_num
    ens_num = "%03d" % i
    filename = HAZ_FILEN.replace('{ens_num}', ens_num)
    dg_fileps[i] = filename

    # Get a xarray.DataArray of the depth grid
    rift_filep = join(HAZ_DIR_UZ, filename)
    ens_dg = rio.open_rasterio(rift_filep).rio.write_crs(HAZ_CRS, inplace=True)

    # Sample depths for the NSI points
    nsi_depth = unexp.pnt_sample_depths(ens_dg,
                                        nsi_clip_out,
                                        'fd_id',
                                        ens_num)
    
    # And Philly
    phil_depth = unexp.pnt_sample_depths(ens_dg,
                                         phil_cent_reproj,
                                         'bfid',
                                         ens_num)
    
    # Add entries to dicts
    # Filter for exposure of at least 1 cm
    nsi_depths[ens_num] = nsi_depth[nsi_depth > .01]
    phil_depths[ens_num] = phil_depth[phil_depth > .01]

    if i % 10 == 0:
        print('Processed ensemble number ' + ens_num)

# Convert to dataframes and write out files
nsi_depths_df = pd.DataFrame.from_dict(nsi_depths)
phil_depths_df = pd.DataFrame.from_dict(phil_depths)

nsi_depths_filep = join(EXP_DIR_I, FIPS, 'nsi_depths.pqt')
phil_depths_filep = join(EXP_DIR_I, FIPS, 'phil_depths.pqt')
unfile.prepare_saving(nsi_depths_filep)
nsi_depths_df.reset_index().to_parquet(nsi_depths_filep)
phil_depths_df.reset_index().to_parquet(phil_depths_filep)

## Generate ensembles

In [ ]:
nsi_clip_out = gpd.read_file(join(EXP_DIR_I, FIPS, 'nsi_res.gpkg'))
phil_inv_out = gpd.read_file(join(EXP_DIR_I, FIPS, 'phil_res.gpkg'))

nsi_depths_filep = join(EXP_DIR_I, FIPS, 'nsi_depths.pqt')
phil_depths_filep = join(EXP_DIR_I, FIPS, 'phil_depths.pqt')

nsi_depths_df = pd.read_parquet(nsi_depths_filep).set_index('fd_id')
phil_depths_df = pd.read_parquet(phil_depths_filep).set_index('bfid')

phil_refs = pd.read_parquet(join(EXP_DIR_I, FIPS, 'phil_ref.pqt'))
nsi_refs = pd.read_parquet(join(EXP_DIR_I, FIPS, 'nsi_ref.pqt'))

In [ ]:
# # Get perimeter estimate with projected CRS
# phil_inv_out['perimeter'] = phil_inv_out['geometry'].to_crs(epsg='5070').length
# phil_inv_out.groupby(['ddf_stories', 'b_type', 'occ_type'])['perimeter'].describe().astype(int).to_csv(join(ABS_DIR, 'perim_summ.csv'))

In [ ]:
# Reference of nsi records linked to philly footprints
lnk_nsi_loc = gpd.sjoin(nsi_clip_out,
                        phil_inv_out,
                        predicate='within',
                        how='inner')
# Reference of unlinked records
unlnk_nsi = nsi_clip_out[~nsi_clip_out['fd_id'].isin(lnk_nsi_loc['fd_id'])]

In [ ]:
# Pass in the full structure dataframe
# Must have the columns: found_type, val_struct, 
# tract_id, num_story, occtype
# Must update est_naccs_loss to account for occtype
# in using the correct DDFs
# We are going to treat all 2+ story properties
# as 2 num_story, and this should link to 
# RES3A-3SNB DDF... (can do logic on RES3A-1SNB)
# This happens in the ensemble generation
# if RES3A occtype -- which is fixed right now --
# then if 1S, RES3A-1SNB and if > 1s choose
# RES3A-3SNB

# Pass in the depth dataframe (in our case, start with a subset of one column)
# Pass in the ref dataframe ()

In [ ]:
# Some updates to phil_inv_out for generating the ensemble
# First, update all ddf_stories from 2 to 3 if occ_type is RES3
# Next, update all RES3 to RES3A to match with NACCS DDFs
# Next, update column names
# occtype, num_story, and fnd_type (just B or S)
phil_inv_ens = phil_inv_out.copy()

phil_inv_ens = phil_inv_ens.merge(phil_refs[['bfid', 'tract_id']],
                                  on='bfid')

phil_inv_ens['num_story'] = phil_inv_ens['ddf_stories'].astype(int)
phil_inv_ens['occtype'] = np.where(phil_inv_ens['occ_type'] == 'RES3',
                                   'RES3A',
                                   'RES1')
phil_inv_ens['found_type'] = np.where(phil_inv_ens['b_type'] == 'WB',
                                    'B',
                                    'S')

# Exclude many stories buildings that
# inflate damage estimates because the
# value at risk is too high
phil_inv_ens = phil_inv_ens[phil_inv_ens['stories_n'].isin(['1', '1.5', '2', '2+', '3'])]

keep_cols = ['bfid', 'val_struct', 'occtype',
             'found_type', 'num_story', 'tract_id']
phil_inv_ens = phil_inv_ens[keep_cols].set_index('bfid')

In [ ]:
# Update the nsi dataset to get the ensembles

keep_cols = ['fd_id', 'val_struct', 'occtype', "found_ht",
             'found_type', 'num_story', 'tract_id']

nsi_inv_ens = nsi_clip_out.merge(nsi_refs[['fd_id', 'tract_id']],
                                 on='fd_id')

# Exclude many stories buildings that
# inflate damage estimates because the
# value at risk is too high
nsi_inv_ens = nsi_inv_ens[nsi_inv_ens['num_story'] <= 3]


nsi_inv_ens['num_story'] = np.where(nsi_inv_ens['num_story'] == 1,
                                    1,
                                    2)
nsi_inv_ens['occtype'] = nsi_inv_ens['occtype'].str[:4].copy()

nsi_inv_ens['occtype'] = np.where(nsi_inv_ens['occtype'] == 'RES3',
                                  'RES3A',
                                  'RES1')

# All no basement are slab for now
nsi_inv_ens['found_type'] = np.where(nsi_inv_ens['found_type'] == 'B',
                                     'B',
                                     'S')

nsi_inv_ens = nsi_inv_ens[keep_cols].set_index('fd_id')

# An all RES3 inventory
# nsi_inv_res3 = nsi_inv_ens.copy()
# nsi_inv_res3['occtype'] = 'RES3A'

In [ ]:
def strct_summ_stats(primary_df, reference_df, strct_col, id_col='tract_id'):
    """
    Create statistical distributions of building characteristics for census tracts.
    
    Parameters:
    -----------
    primary_df : DataFrame
        The primary dataframe containing the characteristic to analyze
    reference_df : DataFrame
        The reference dataframe that may contain additional tracts not in primary_df
    strct_col : str
        The column name of the categorical characteristic to analyze (e.g., 'found_type', 'num_story')
    id_col : str, default='tract_id'
        The column name for the geographic identifier (e.g., census tract)
        
    Returns:
    --------
    DataFrame
        A dataframe with the distribution of the characteristic for all tracts in both dataframes,
        using tract-specific distributions where available and overall averages for tracts missing
        from the primary_df that we draw the tract-level distributions from.
    """
    # Get all unique tract IDs from both dataframes
    all_tracts = pd.concat([
        primary_df[id_col], 
        reference_df[id_col]
    ]).unique()
    
    # Find tracts in reference_df that are missing from primary_df
    missing_tracts = reference_df[~reference_df[id_col].isin(primary_df[id_col])][id_col].unique()
    
    # Calculate tract-level distributions for the characteristic in primary_df
    char_sum = primary_df.groupby([id_col, strct_col]).size()
    char_prop = char_sum / primary_df.groupby(id_col).size()
    
    # Convert to a pivot table with tracts as rows and characteristic values as columns
    char_stats = (
        char_prop.reset_index()
        .pivot(index=id_col, columns=strct_col, values=0)
        .fillna(0)
    )
    
    # Calculate the overall distribution across all records in primary_df
    overall_dist = primary_df.groupby(strct_col).size() / len(primary_df)
    
    # Get the unique values of the characteristic
    char_values = overall_dist.index.tolist()
    
    # Create a dataframe with the overall distribution for each missing tract
    n_missing = len(missing_tracts)
    
    if n_missing > 0:
        # Create a matrix of the overall distribution repeated for each missing tract
        fill_df = pd.DataFrame(
            np.repeat([overall_dist.values], n_missing, axis=0),
            index=missing_tracts,
            columns=overall_dist.index
        )
        
        # Combine with the tract-specific distributions
        char_stats = pd.concat([char_stats, fill_df], axis=0)
    
    # Ensure all columns exist (in case some characteristic values don't appear in some tracts)
    for val in char_values:
        if val not in char_stats.columns:
            char_stats[val] = 0
    
    return char_stats

In [ ]:
# Get phil & nsi summary stats for census tracts
phil_found_stats = strct_summ_stats(phil_inv_ens, nsi_inv_ens, 'found_type')
phil_stories_stats = strct_summ_stats(phil_inv_ens, nsi_inv_ens, 'num_story')

nsi_found_stats = strct_summ_stats(nsi_inv_ens, phil_inv_ens, 'found_type')
nsi_stories_stats = strct_summ_stats(nsi_inv_ens, phil_inv_ens, 'num_story')

# Update all occtype to res3 as a one-at-a-time analysis
nsi_inv_res3 = nsi_inv_ens.copy()
nsi_inv_res3['occtype'] = 'RES3A'

In [ ]:
dg_id = '009'
d_min = 0

# Define base configurations for both basement adjustment options
base_configs = {
    'no_adj': {
        'coef_var': COEF_VARIATION,
        'n_sow': 1000,
        'id_col': 'fd_id',
        'base_adj': False,
        'color': '#0077BB',
        'label': 'Status Quo Approach',
        'depth_min': d_min
    },
    # 'with_adj': {
    #     'coef_var': COEF_VARIATION,
    #     'id_col': 'fd_id',
    #     'n_sow': 100,
    #     'base_adj': True,
    #     'color': '#EE7733',
    #     'depth_min': d_min,
    #     'label': 'Basement Adjustment Approach'
    # }
}

# Define configurations as variations from base
model_configs = {
    'phil': {
        'struct_list': ['val_struct', 'ffe'],
        'id_col': 'bfid',
        'inventory': phil_inv_ens,
        'depths': phil_depths_df[[dg_id]],
        'label': 'Philly Loc., Val, FFE',
    },
    # 'nsi_ddfs': {
    #     'struct_list': [],
    #     'id_col': 'fd_id',
    #     'inventory': nsi_inv_ens,
    #     'depths': nsi_depths_df[[dg_id]],
    #     'label': 'NSI Loc., DDF Uncertainty Only',
    # },
    # 'nsi_valffe': {
    #     'struct_list': ['val_struct', 'ffe'],
    #     'id_col': 'fd_id',
    #     'inventory': nsi_inv_ens,
    #     'depths': nsi_depths_df[[dg_id]],
    #     'label': 'NSI Loc., Val, FFE',
    # },
    # 'nsi_valffest': {
    #     'struct_list': ['val_struct', 'ffe', 'num_story'],
    #     'id_col': 'fd_id',
    #     'inventory': nsi_inv_ens,
    #     'depths': nsi_depths_df[[dg_id]],
    #     'label': 'NSI Loc., Val, FFE, Stories',
    # },
    # 'nsi_valffefound': {
    #     'struct_list': ['val_struct', 'ffe', 'found_type'],
    #     'id_col': 'fd_id',
    #     'inventory': nsi_inv_ens,
    #     'depths': nsi_depths_df[[dg_id]],
    #     'label': 'NSI Loc., Val, FFE, Basement',
    # },
    # 'nsi_unsafe': {
    #     'struct_list': ['val_struct', 'ffe', 'num_story', 'found_type'],
    #     'id_col': 'fd_id',
    #     'inventory': nsi_inv_ens,
    #     'depths': nsi_depths_df[[dg_id]],
    #     'label': 'NSI Loc., All Uncertain',
    # },
    # 'nsi_phil': {
    #     'struct_list': ['val_struct', 'ffe', 'num_story', 'found_type'],
    #     'id_col': 'fd_id',
    #     'inventory': nsi_inv_ens,
    #     'depths': nsi_depths_df[[dg_id]],
    #     'found_param': phil_found_stats,
    #     'stories_param': phil_stories_stats,
    #     'label': 'NSI Loc., Philly Dist. All Uncertain',
    # },
    # 'phil_unsafe': {
    #     'struct_list': ['val_struct', 'ffe', 'num_story', 'found_type'],
    #     'id_col': 'bfid',
    #     'inventory': phil_inv_ens,
    #     'depths': phil_depths_df[[dg_id]],
    #     'label': 'Philly Loc., Philly Dist. All Uncertain',
    # },
    # 'phil_nsi': {
    #     'struct_list': ['val_struct', 'ffe', 'num_story', 'found_type'],
    #     'id_col': 'bfid',
    #     'inventory': phil_inv_ens,
    #     'depths': phil_depths_df[[dg_id]],
    #     'stories_param': nsi_stories_stats,
    #     'found_param': nsi_found_stats,
    #     'label': 'Philly Loc., NSI Dist. All Uncertain',
    # },
    # 'nsi_val': {
    #     'struct_list': ['val_struct'],
    #     'id_col': 'fd_id',
    #     'inventory': nsi_inv_ens,
    #     'depths': nsi_depths_df[[dg_id]],
    #     'label': 'NSI - Vary Value Only',
    # },
    # 'nsi_res3': {
    #     'struct_list': ['val_struct', 'ffe', 'num_story', 'found_type'],
    #     'id_col': 'fd_id',
    #     'inventory': nsi_inv_res3,
    #     'depths': nsi_depths_df[[dg_id]],
    #     'label': 'NSI Loc w/ RES3',
    # },
    # 'nsi_philres3': {
    #     'struct_list': ['val_struct', 'ffe', 'num_story', 'found_type'],
    #     'id_col': 'fd_id',
    #     'inventory': nsi_inv_res3,
    #     'depths': nsi_depths_df[[dg_id]],
    #     'found_param': phil_found_stats,
    #     'stories_param': phil_stories_stats,
    #     'label': 'NSI Loc w/ RES3 & Phil Dist',
    # },
}

In [ ]:
# Run all simulations for both basement adjustment options
results = {}
for adj_name, base_config in base_configs.items():
    for model_name, model_config in model_configs.items():
        name = f"{model_name}:{adj_name}"
        
        # Combine configurations
        run_config = {**base_config}
        for key in ['struct_list', 'id_col', 'found_param', 'stories_param']:
            if key in model_config:
                run_config[key] = model_config[key]
        
        # Run the simulation
        results[name] = unens.get_loss_ensemble(
            model_config['inventory'],
            model_config['depths'],
            config=run_config,
            vuln_dir=VULN_DIR_I,
            # random_seed=base_config['random_seed']
        )

dam_col = 'naccs_loss_' + dg_id

# Run benchmarks for both basement adjustment options
benchmarks = {}
for adj_name, base_config in base_configs.items():
    benchmarks[adj_name] = unens.benchmark_naccs_loss(
        nsi_inv_ens,
        nsi_depths_df[[dg_id]],
        VULN_DIR_I,
        base_adj=base_config['base_adj'],
        depth_min=base_config['depth_min'],
    )
    benchmarks[adj_name + '_sum'] = benchmarks[adj_name][dam_col].sum()/1e8

In [ ]:
# Divide results into subsets 
# There are specific analyses we want to do on
# NSI records linked to Philly buildings
# We also want to contextualize the losses
# for NSI & Philly records not linked to one another

# Create dictionaries to store linked and unlinked results
linked_results = {}
unlinked_results = {}

for name, result in results.items():
    model_name, adj_name = name.split(':', 1)

    # Create linked subset
    link_id = model_configs[model_name]['id_col']
    linked_subset = result[result[link_id].isin(lnk_nsi_loc[link_id])].copy()
    linked_key = f"{model_name}_linked:{adj_name}"
    linked_results[linked_key] = linked_subset

    # Create unlinked subset
    unlinked_subset = result[~result[link_id].isin(lnk_nsi_loc[link_id])].copy()
    unlinked_key = f"{model_name}_unlinked:{adj_name}"
    unlinked_results[unlinked_key] = unlinked_subset

# Combine all results dictionaries for easier access
all_results = {
    'original': results,
    'linked': linked_results,
    'unlinked': unlinked_results
}

# Process benchmarks to create linked and unlinked versions
linked_benchmarks = {}
unlinked_benchmarks = {}

# Get the list of linked and unlinked fd_ids
linked_fd_ids = set(lnk_nsi_loc['fd_id'])
unlinked_fd_ids = set(unlnk_nsi['fd_id'])

for adj_name, benchmark in benchmarks.items():
    # Skip the '_sum' entries which are just scalar values
    if '_sum' in adj_name:
        continue
    
    # Create linked subset
    linked_subset = benchmark[benchmark.index.isin(linked_fd_ids)].copy()
    linked_key = f"{adj_name}_linked"
    linked_benchmarks[linked_key] = linked_subset
    linked_benchmarks[linked_key + '_sum'] = linked_subset[dam_col].sum()/1e8
    
    # Create unlinked subset
    unlinked_subset = benchmark[benchmark.index.isin(unlinked_fd_ids)].copy()
    unlinked_key = f"{adj_name}_unlinked"
    unlinked_benchmarks[unlinked_key] = unlinked_subset
    unlinked_benchmarks[unlinked_key + '_sum'] = unlinked_subset[dam_col].sum()/1e8

# Combine all benchmarks dictionaries
all_benchmarks = {
    'original': benchmarks,
    'linked': linked_benchmarks,
    'unlinked': unlinked_benchmarks
}

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import seaborn as sns

# Process results for plotting
processed_data = []
for name, result in results.items():
    model_name, adj_name = name.split(':', 1)
    
    if adj_name == 'no_adj':
        # Calculate total loss per sow
        loss = result.groupby('sow_ind')[dam_col].sum()/1e8
        
        # Prepare for boxplot
        df = loss.reset_index()
        df['Structure Data'] = model_configs[model_name]['label']
        df['Basement Adjustment'] = base_configs[adj_name]['label']
        processed_data.append(df)

# Combine all data for boxplot
combined_data = pd.concat(processed_data, axis=0)

# Create figure
fig, ax = plt.subplots(figsize=(10, 6), dpi=300)

# Create boxplot with structure data on y-axis and hue for basement adjustment
sns.boxplot(
    ax=ax,
    data=combined_data,
    y='Structure Data',
    x=dam_col,
    hue='Basement Adjustment',
    showmeans=True,
    meanprops={'markerfacecolor': 'firebrick', 'markeredgecolor': 'black', 'marker': 'D'},
    palette=[base_configs['no_adj']['color']]#, base_configs['with_adj']['color']]
)

# Add benchmark lines with matching colors
for run_title, base_config in base_configs.items():
    if 'no_adj' in run_title:
        ax.axvline(
            benchmarks[run_title + '_sum'], 
            color=base_config['color'], 
            linestyle='dashed' if 'RES3A' in run_title else 'dashdot', 
            linewidth=2, 
            label=f"NSI Benchmark ({base_config['label']})"
        )

# Format plot
ax.set_xlabel('Hurricane Irene Damages ($ 100 Millions)', size=16)
ax.set_ylabel('')  # No y-axis label needed
ax.tick_params(labelsize=14)
ax.grid(axis='x', linestyle='--', alpha=0.7)

# Customize legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles=handles, 
    labels=labels,
    fontsize=14,
    bbox_to_anchor=(.38, -.25),
    loc='center',
    frameon=True,
    framealpha=0.9,
    ncols=2,
)

# Add a background color to alternate rows for better visual separation
num_categories = len(combined_data['Structure Data'].unique())
for i in range(num_categories):
    if i % 2 == 0:  # Every other row
        ax.axhspan(i-0.5, i+0.5, color='lightgray', alpha=0.2, zorder=0)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Process results for plotting
processed_data = []
for name, result in results.items():
    model_name, adj_name = name.split(':', 1)
    
    if adj_name == 'no_adj':

        # Calculate total loss per sow
        loss = result.groupby('sow_ind')[dam_col].sum()/1e8
        
        # Prepare for pointplot
        for value in loss:
            processed_data.append({
                'Structure Data': model_configs[model_name]['label'],
                # 'Basement Adjustment': base_configs[adj_name]['label'],
                'Loss': value
            })

# Convert to DataFrame
plot_data = pd.DataFrame(processed_data)

# Create figure
fig, ax = plt.subplots(figsize=(10, 6), dpi=300)

# Create pointplot with means and std error
sns.pointplot(
    ax=ax,
    data=plot_data,
    y='Structure Data',
    x='Loss',
    # hue='Basement Adjustment',
    # palette=[base_configs['no_adj']['color'], base_configs['with_adj']['color']],
    # markers=['o', 's'],
    # markersize=10,
    estimator='mean',
    errorbar=('ci', 99),
    # dodge=.5,
    ls=''
)

# Add benchmark lines with matching colors
for run_title, base_config in base_configs.items():
    if 'no_adj' in run_title:
        ax.axvline(
            benchmarks[run_title + '_sum'],
            color=base_config['color'],
            linestyle='dashed' if 'RES3A' in run_title else 'dashdot',
            linewidth=2,
            label=f"NSI Benchmark ({base_config['label']})"
        )

# Format plot
ax.set_xlabel('Hurricane Irene Damages ($ 100 Millions)', size=16)
ax.set_ylabel('') 
ax.tick_params(axis='both', labelsize=14)

# Add grid lines
ax.grid(axis='x', linestyle='--', alpha=0.7)
ax.grid(axis='y', linestyle='-', linewidth=1.5, alpha=0.2)

# Add a background color to alternate rows for better visual separation
num_categories = len(plot_data['Structure Data'].unique())
for i in range(num_categories):
    if i % 2 == 0:
        ax.axhspan(i-0.5, i+0.5, color='lightgray', alpha=0.2, zorder=0)

# Customize legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles=handles,
    labels=labels,
    fontsize=14,
    loc='center',
    frameon=True,
    framealpha=0.9,
    ncol=2,
    bbox_to_anchor=(.31, -.3)
)

plt.tight_layout()

In [ ]:
def create_comparison_dataframe(results_dict,
                                benchmark_dict,
                                phil_refs,
                                nsi_refs, 
                                phil_inventory,
                                nsi_inventory, 
                                dam_col,
                                ref_id,
                                result_keys=['phil:no_adj'],
                                benchmark_keys=['no_adj']):
    """
    Create a dataframe that links building IDs to different reference IDs
    for both Philadelphia and NSI data and aggregates damage and value
    to the level of the reference ID.
    
    Parameters:
    -----------
    results_dict : dict
        Dictionary containing ensemble results
    benchmark_dict : dict
        Dictionary containing benchmark results
    phil_refs : DataFrame
        DataFrame linking Philadelphia building IDs to reference IDs
    nsi_refs : DataFrame
        DataFrame linking NSI building IDs to reference IDs
    phil_inventory : DataFrame
        Philadelphia inventory data
    nsi_inventory : DataFrame
        NSI inventory data
    dam_col : str
        Column name for damage values
    ref_id: str
        Name of the spatial reference (e.g., "tract_id"). Must be in
        the phil_refs and nsi_refs dataframes
    result_keys : list, default=['phil:no_adj']
        Key to access specific results in results_dict
    benchmark_keys : list, default=['no_adj']
        Key to access specific benchmarks in benchmark_dict
        
    Returns:
    --------
    DataFrame
        Comparison dataframe with damage and property values for both datasets aggregated to
        the level of ref_id
    """
    # Process each result key
    result_dfs = {}
    for key in result_keys:
        # Determine which reference dataframe to use based on key prefix
        if key.startswith('phil'):
            refs = phil_refs
            id_col = 'bfid'
            inventory = phil_inventory
        else:
            refs = nsi_refs
            id_col = 'fd_id'
            inventory = nsi_inventory

        # Process ensemble results
        temp = results_dict[key]
        if ref_id not in temp.columns:
            temp = temp.merge(refs, on=id_col)

        temp_gb = temp.groupby(['sow_ind', ref_id]).agg({dam_col: 'sum'}).reset_index()
        loss_by_ref = temp_gb.groupby(ref_id)[dam_col].mean()
        
        # Calculate reference-level property values
        if ref_id not in inventory.columns:
            inventory = inventory.merge(refs[[id_col, ref_id]], on=id_col)
        
        ref_vals = inventory.groupby(ref_id).agg({'val_struct': ['median', 'sum', 'size']})
        ref_vals = ref_vals.reset_index()
        ref_vals.columns = [ref_id, 'median_val', 'total_val', 'n_prop']
        
        # Create result dataframe
        result_df = pd.DataFrame({
            dam_col: loss_by_ref,
            'median_val': ref_vals.set_index(ref_id)['median_val'],
            'total_val': ref_vals.set_index(ref_id)['total_val'],
            'n_prop': ref_vals.set_index(ref_id)['n_prop']
        })
        
        result_df = result_df[result_df[dam_col].notnull()]
        result_dfs[key] = result_df
   
    # Process each benchmark key
    benchmark_dfs = {}
    for key in benchmark_keys:
        refs = nsi_refs
        id_col = 'fd_id'
        inventory = nsi_inventory

    # Process benchmark results
        temp = benchmark_dict[key].reset_index()
        
        if ref_id not in inventory.columns:
            inventory = inventory.merge(refs[[id_col, ref_id]], on=id_col)
        
        if id_col in temp.columns:
            temp_w_refs = temp.merge(inventory, on=id_col)
            temp_gb = temp_w_refs.groupby([ref_id])[[dam_col]].sum().reset_index()
        else:
            # If benchmark already has ref_id, just group by it
            temp_gb = temp.groupby([ref_id])[[dam_col]].sum().reset_index()
        
        # Calculate reference-level property values
        ref_vals = inventory.groupby(ref_id).agg({'val_struct': ['median', 'sum', 'size']})
        ref_vals = ref_vals.reset_index()
        ref_vals.columns = [ref_id, 'median_val', 'total_val', 'n_prop']
        
        # Merge with property values
        temp_gb = temp_gb.merge(ref_vals, on=ref_id)
        benchmark_dfs[key] = temp_gb.set_index(ref_id)

    # Combine all dataframes
    all_dfs = []
    
    # Process result dataframes
    for key, df in result_dfs.items():
        df_reset = df.reset_index()
        df_reset.columns = [ref_id] + [f"{col}_{key.split(':')[0]}" for col in df.columns]
        all_dfs.append(df_reset)
    
    # Process benchmark dataframes
    for key, df in benchmark_dfs.items():
        df_reset = df.reset_index()
        df_reset.columns = [ref_id] + [f"{col}_nsi" for col in df.columns]
        all_dfs.append(df_reset)

    # Merge all dataframes
    if all_dfs:
        result = all_dfs[0]
        for df in all_dfs[1:]:
            result = result.merge(df, on=ref_id, how='outer')
        
        return result.fillna(0)
    else:
        return pd.DataFrame()


In [ ]:
comp = create_comparison_dataframe(
    all_results['original'], 
    all_benchmarks['original'],
    phil_refs,
    nsi_refs,
    phil_inv_ens,
    nsi_inv_ens,
    dam_col=dam_col,
    ref_id='tract_id',
    result_keys=['phil:no_adj', 'nsi_ddfs:no_adj', 'nsi_unsafe:no_adj',
                 'nsi_phil:no_adj', 'phil_unsafe:no_adj', 'phil_nsi:no_adj'],
    benchmark_keys=['no_adj']
)

In [ ]:
tract_ref = gpd.read_file(join(REF_DIR_I, FIPS, 'tract.gpkg'))[['GEOID', 'geometry']]
comp_geo = tract_ref.merge(comp, left_on='GEOID', right_on='tract_id')

In [ ]:
# import contextily as cx
# from matplotlib import ticker
# import matplotlib.colors as colors
# import matplotlib.gridspec as gridspec
# from mpl_toolkits.axes_grid1 import make_axes_locatable
# import seaborn as sns

# # Create figure
# fig = plt.figure(figsize=(12, 16), dpi=300)

# # Create two GridSpecs
# # Top GridSpec for the maps
# gs_top = gridspec.GridSpec(1, 5)
# gs_top.update(top=0.95, bottom=0.66)  # Control vertical position

# # Bottom GridSpec for the scatterplot and boxplot
# gs_bottom = gridspec.GridSpec(2, 5, height_ratios=[1, 1])
# gs_bottom.update(top=0.65, bottom=0.15)  # Control vertical position and leave space between GridSpecs
# gs_bottom.update(left=.3, right=.7)

# # Create axes for maps
# ax0 = fig.add_subplot(gs_top[0, 0:2])
# ax1 = fig.add_subplot(gs_top[0, 2:4])

# # Create axes for scatterplot and boxplot
# ax3 = fig.add_subplot(gs_bottom[0, 0:4])  # Scatterplot
# ax4 = fig.add_subplot(gs_bottom[1, 0:4])  # Boxplot

# comp_plot = comp_geo.to_crs(epsg=3857)

# cmap = 'YlOrRd'

# comp_plot['nsi_loss'] = comp_plot[dam_col + '_nsi']/1e6
# comp_plot['phil_loss'] = comp_plot[dam_col + '_phil']/1e6

# comp_plot['dam_bias'] = comp_plot['nsi_loss'] - comp_plot['phil_loss']
# comp_plot['rank_bias'] = (comp_plot['nsi_loss'].rank(ascending=False) -
#                           comp_plot['phil_loss'].rank(ascending=False))

# total_nsi_loss = comp_plot['nsi_loss'].sum()
# total_phil_loss = comp_plot['phil_loss'].sum()

# vmin = min(comp_plot['nsi_loss'].min(), comp_plot['phil_loss'].min())
# vmax = max(comp_plot['nsi_loss'].max(), comp_plot['phil_loss'].max())
# vcenter=1

# comp_plot.plot(ax=ax0, column='phil_loss', cmap='Reds',
#                legend=True,
#                legend_kwds={'pad': .03,
#                            'shrink': .75},
#                vmin=0, vmax=comp_plot['phil_loss'].max())
# comp_plot.plot(ax=ax1, column='dam_bias', cmap='bwr',
#                legend=True,
#                legend_kwds={'pad': .03,
#                            'shrink': .75},
#                norm=colors.TwoSlopeNorm(vmin=comp_plot['dam_bias'].min(),
#                                         vcenter=0,
#                                         vmax=comp_plot['dam_bias'].max()))

# cx.add_basemap(ax0,
#                attribution_size=4,
#                source=cx.providers.Esri.WorldImagery)

# cx.add_basemap(ax1,
#                attribution_size=4,
#                source=cx.providers.Esri.WorldImagery)


# axes = [ax0, ax1]
# for i in range(len(axes)):
#     axes[i].tick_params(
#         axis="both",
#         which="both",
#         bottom=False,
#         left=False,
#         labelbottom=False,
#         labelleft=False,
#     )

# # Add titles
# axes[0].set_title("Damages ($ M) w/\nPhilly Structures", fontsize=14)
# axes[1].set_title("Deviation ($ M) Introduced by\nNational Structure Inventory", fontsize=14)

# # Add annotations
# axes[0].annotate('Total = ${} M'.format(int(total_phil_loss)),
#              xy=(.05, .94),
#              xycoords='axes fraction',
#              color='black',
#              backgroundcolor='white',
#              size=14
#              )

# axes[1].annotate('Deviation = ${} M'.format(int(total_nsi_loss - total_phil_loss)),
#              xy=(.05, .94),
#              xycoords='axes fraction',
#              color='black',
#              backgroundcolor='white',
#              size=14
#              )

# # Update colorbar tick label size
# for ax in fig.axes:
#     if ax._axes.get_label() == '<colorbar>':
#         ax.tick_params(labelsize='14')

# comp_plot['val_thou'] = comp_plot['median_val']/1e3
# comp_plot['phil_rank'] = comp_plot['phil_loss'].rank(ascending=False)
# comp_plot['nsi_rank'] = comp_plot['nsi_loss'].rank(ascending=False)

# sns.scatterplot(
#     ax=ax3,
#     y='nsi_rank',
#     x='phil_rank',
#     edgecolor='gray',
#     hue='dam_bias',
#     palette='bwr',
#     norm=colors.TwoSlopeNorm(vmin=comp_plot['dam_bias'].min(),
#                                         vcenter=0,
#                                         vmax=comp_plot['dam_bias'].max()),
#     data=comp_plot,
#     legend=False
# )

# ax3.axline([0, 0], [1, 1], color='black', linestyle='--', alpha=0.5, zorder=0)

# ax3.set_xlabel('Damage Rank with Philly Structures', fontsize=14)
# ax3.set_ylabel('Damage Rank with\nNational Structure Inventory', fontsize=14)
# ax3.tick_params(labelsize=14)

# ax3.axvline(20, color='gray', linestyle='--', alpha=.2)
# ax3.axhline(20, color='gray', linestyle='--', alpha=.2)

# ax3.annotate('Top 20th% Type 1 Error',
#              (75, 1.5),
#              xycoords='data',
#              ha='center',
#              size=12
#              )

# ax3.annotate('Top\n20th%\nType 2\nError',
#              (8, 62),
#              xycoords='data',
#              ha='center',
#              size=12
#              )


# comp_plot['val_bins'] = pd.qcut(comp_plot['val_thou'], q=5)
# comp_plot["val_bins"] = comp_plot["val_bins"].apply(lambda x: pd.Interval(left=int(round(x.left)), right=int(round(x.right))))

# comp_plot['nsi_rel_loss'] = (comp_plot['nsi_loss']*1e6)/(comp_plot['total_val_nsi'])
# comp_plot['phil_rel_loss'] = (comp_plot['phil_loss']*1e6)/(comp_plot['total_val_phil'])

# temp1 = comp_plot[['val_bins', 'val_thou', 'nsi_rel_loss', 'nsi_loss']]
# temp1.columns = ['val_bins', 'val_thou', 'rel_loss', 'loss']
# temp1['Inventory'] = 'NSI'
# temp2 = comp_plot[['val_bins', 'val_thou', 'phil_rel_loss', 'phil_loss']]
# temp2.columns = ['val_bins', 'val_thou', 'rel_loss', 'loss']
# temp2['Inventory'] = 'Philly'
# temp = pd.concat([temp1, temp2], axis=0)
# comp_plot['rel_diff'] = comp_plot['nsi_rel_loss'] - comp_plot['phil_rel_loss']
# comp_plot['diff'] = comp_plot['nsi_loss'] - comp_plot['phil_loss']


# sns.boxplot(
#     ax=ax4,
#     y='rel_loss',
#     x='val_bins',
#     hue='Inventory',
#     showfliers=False,
#     showmeans=True,
#     meanprops={'markerfacecolor': 'firebrick', 'markeredgecolor': 'black', 'marker': 'D'},
#     data=temp
# )
# ax4.set_ylabel('Tract Damages Normalized\nby Total Structure Value', size=14)
# ax4.set_xlabel('Quintile of Tract Median Structure Value ($ K)', size=14)
# ax4.tick_params('x', rotation=15)
# ax4.tick_params('both', labelsize=12)

# # For the boxplot legend, move it outside to the right
# ax4.legend(
#     bbox_to_anchor=(.5, 0.91),
#     loc='center left',
#     fontsize=12,
#     frameon=True
# )

In [ ]:
# comp_bg = create_comparison_dataframe(
#     all_results['original'], 
#     all_benchmarks['original'],
#     phil_refs,
#     nsi_refs,
#     phil_inv_ens,
#     nsi_inv_ens,
#     dam_col=dam_col,
#     ref_id='bg_id',
#     result_key='phil:no_adj',
#     benchmark_key='no_adj'
# )

# comp_bg = comp_bg[comp_bg['n_prop'] > 5]

# comp_bg['nsi_loss'] = comp_bg[dam_col + '_nsi']/1e6
# comp_bg['phil_loss'] = comp_bg[dam_col + '_phil']/1e6

# comp_bg['dam_bias'] = comp_bg['nsi_loss'] - comp_bg['phil_loss']
# comp_bg['rank_bias'] = (comp_bg['nsi_loss'].rank(ascending=False) -
#                           comp_bg['phil_loss'].rank(ascending=False))

# comp_bg['val_thou'] = comp_bg['median_val']/1e3
# comp_bg['phil_rank'] = comp_bg['phil_loss'].rank(ascending=False)
# comp_bg['nsi_rank'] = comp_bg['nsi_loss'].rank(ascending=False)

# comp_bg['val_bins'] = pd.qcut(comp_bg['val_thou'], q=5)
# comp_bg["val_bins"] = comp_bg["val_bins"].apply(lambda x: pd.Interval(left=int(round(x.left)), right=int(round(x.right))))

# comp_bg['nsi_rel_loss'] = (comp_bg['nsi_loss']*1e6)/(comp_bg['total_val_nsi'])
# comp_bg['phil_rel_loss'] = (comp_bg['phil_loss']*1e6)/(comp_bg['total_val_phil'])

# temp1 = comp_bg[['val_bins', 'val_thou', 'nsi_rel_loss', 'nsi_loss']]
# temp1.columns = ['val_bins', 'val_thou', 'rel_loss', 'loss']
# temp1['Inventory'] = 'NSI'
# temp2 = comp_bg[['val_bins', 'val_thou', 'phil_rel_loss', 'phil_loss']]
# temp2.columns = ['val_bins', 'val_thou', 'rel_loss', 'loss']
# temp2['Inventory'] = 'Philly'
# temp = pd.concat([temp1, temp2], axis=0)
# comp_bg['rel_diff'] = comp_bg['nsi_rel_loss'] - comp_bg['phil_rel_loss']
# comp_bg['diff'] = comp_bg['nsi_loss'] - comp_bg['phil_loss']

# fig, ax = plt.subplots(dpi=300)
# # sns.barplot(
# #     ax=ax,
# #     y='rel_loss',
# #     x='val_bins',
# #     hue='Inventory',
# #     data=temp
# # )

# # sns.boxplot(
# #     ax=ax,
# #     y='rel_loss',
# #     x='val_bins',
# #     hue='Inventory',
# #     showfliers=False,
# #     showmeans=True,
# #     meanprops={'markerfacecolor': 'firebrick', 'markeredgecolor': 'black', 'marker': 'D'},
# #     data=temp
# # )

# sns.boxplot(
#     ax=ax,
#     y='rank_bias',
#     x='val_bins',
#     showfliers=False,
#     showmeans=True,
#     meanprops={'markerfacecolor': 'firebrick', 'markeredgecolor': 'black', 'marker': 'D'},
#     data=comp_bg
# )

# # sns.violinplot(
# #     ax=ax,
# #     y='rel_loss',
# #     x='val_bins',
# #     hue='Inventory',
# #     inner='quart',
# #     split=True,
# #     data=temp
# # )

# # sns.lmplot(
# #     y='diff',
# #     x='val_thou',
# #     data=comp_bg
# # )

# ax.set_ylabel('Tract Damages Normalized\nby Total Structure Value', size=14)
# ax.set_xlabel('Tract Median Structure Value ($ K)', size=14)
# ax.tick_params('x', rotation=15)
# ax.tick_params('both', labelsize=12)

# # For the boxplot legend, move it outside to the right
# ax.legend(
#     loc='upper left',
#     fontsize=12,
#     frameon=True
# )

In [ ]:
import contextily as cx
from matplotlib import ticker
import matplotlib.colors as colors
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable
import seaborn as sns

# Create figure
fig = plt.figure(figsize=(12, 12), dpi=300)

# Create two GridSpecs
# Top GridSpec for the maps
gs = gridspec.GridSpec(2, 5, height_ratios=[3, 2],
                       hspace=.03)

# Create axes for maps
ax0 = fig.add_subplot(gs[0, 0:2])
ax1 = fig.add_subplot(gs[0, 2:4])

# Create axes for scatterplot and boxplot
ax3 = fig.add_subplot(gs[1, 1:3])  # Scatterplot

comp_plot = comp_geo.to_crs(epsg=3857)

cmap = 'YlOrRd'

comp_plot['nsi_loss'] = comp_plot[dam_col + '_nsi']/1e6
comp_plot['phil_loss'] = comp_plot[dam_col + '_phil']/1e6

comp_plot['dam_bias'] = comp_plot['nsi_loss'] - comp_plot['phil_loss']
comp_plot['rank_bias'] = (comp_plot['nsi_loss'].rank(ascending=False) -
                          comp_plot['phil_loss'].rank(ascending=False))

comp_plot['dam_pct_bias'] = 100*comp_plot['dam_bias']/comp_plot['phil_loss']+.01

total_nsi_loss = comp_plot['nsi_loss'].sum()
total_phil_loss = comp_plot['phil_loss'].sum()
total_pct_dev = np.round(100*(total_nsi_loss-total_phil_loss)/total_phil_loss)

vmin = min(comp_plot['nsi_loss'].min(), comp_plot['phil_loss'].min())
vmax = max(comp_plot['nsi_loss'].max(), comp_plot['phil_loss'].max())
vcenter=1

comp_plot[comp_plot['phil_loss'] > 0].plot(ax=ax0, column='phil_loss', cmap='Reds',
               legend=True,
               legend_kwds={'pad': .03,
                           'shrink': .75},
               vmin=0, vmax=comp_plot['phil_loss'].max())
comp_plot[comp_plot['phil_loss'] > 0].plot(ax=ax1, column='dam_pct_bias', cmap='bwr',
               legend=True,
               legend_kwds={'pad': .03,
                           'shrink': .75,
                           'extend': 'max'},
               norm=colors.TwoSlopeNorm(vmin=-60,
                                        vcenter=0,
                                        vmax=100))

cx.add_basemap(ax0,
               attribution_size=4,
               source=cx.providers.Esri.WorldImagery)

cx.add_basemap(ax1,
               attribution_size=4,
               source=cx.providers.Esri.WorldImagery)


axes = [ax0, ax1]
for i in range(len(axes)):
    axes[i].tick_params(
        axis="both",
        which="both",
        bottom=False,
        left=False,
        labelbottom=False,
        labelleft=False,
    )

# Add titles
axes[0].set_title("Damages ($ M) w/\nPhilly Structures", fontsize=14)
axes[1].set_title("Discrepancy (%) Introduced by\nNational Structure Inventory", fontsize=14)

# Add annotations
axes[0].annotate('Total = ${} M'.format(int(total_phil_loss)),
             xy=(.04, .95),
             xycoords='axes fraction',
             color='black',
             backgroundcolor='white',
             size=14
             )

axes[1].annotate('Total = {} %'.format(int(total_pct_dev)),
             xy=(.04, .95),
             xycoords='axes fraction',
             color='black',
             backgroundcolor='white',
             size=14
             )

# Update colorbar tick label size
for ax in fig.axes:
    if ax._axes.get_label() == '<colorbar>':
        ax.tick_params(labelsize='14')

comp_plot['val_thou'] = comp_plot['median_val_phil']/1e3
comp_plot['phil_rank'] = comp_plot['phil_loss'].rank(ascending=False)
comp_plot['nsi_rank'] = comp_plot['nsi_loss'].rank(ascending=False)

sns.scatterplot(
    ax=ax3,
    y='nsi_rank',
    x='phil_rank',
    edgecolor='gray',
    hue='dam_pct_bias',
    palette='bwr',
    hue_norm=colors.TwoSlopeNorm(vmin=-60,
                                        vcenter=0,
                                        vmax=100),
    data=comp_plot.sort_values('phil_rank', ascending=False),
    legend=False
)

ax3.axline([0, 0], [1, 1], color='black', linestyle='--', alpha=0.5, zorder=0)

ax3.set_xlabel('Damage Rank with Philly Structures', fontsize=14)
ax3.set_ylabel('Damage Rank with\nNational Structure Inventory', fontsize=14)
ax3.tick_params(labelsize=14)

ax3.annotate('Top 10th Percentile\nType 1 Error',
             (18, 1),
             xycoords='data',
             ha='center',
             color='red',
             size=12
             )

ax3.annotate('Top 10th Percentile\nType 2 Error',
             (5.2, 20.5),
             xycoords='data',
             ha='center',
             color='blue',
             size=12
             )

ax3.set_xlim([0, 25])
ax3.set_ylim([0, 23])

ax3.axvline(11, ymin=11/23., color='blue', linestyle='--', alpha=1, zorder=0)
ax3.axhline(11, xmax=11/25., color='blue', linestyle='--', alpha=1, zorder=0)
ax3.axvline(11, ymax=11/23., color='red', linestyle='--', alpha=1, zorder=0)
ax3.axhline(11, xmin=11/25., color='red', linestyle='--', alpha=1, zorder=0)

In [ ]:
len(comp_plot[comp_plot['dam_pct_bias'] < 0])/len(comp_plot)

In [ ]:
# Loop through experiments to do rank order comparisons
# Might make sense to just calculate a lot of metrics like the agg bias paper
# and do small multiples comparing the different estimation approaches 
# I think small multiples makes most sense. A few ideas
# Total discrepancy ($), Total discrepancy (%),
# RMSE Matched Properties, RMSE Tract
# Rank Order Correlation Matched Properties, '' Tract
# Rank Order Correlation top 10% tracts, top 20% tracts,
# exps = ['nsi', 'nsi_ddfs', 'nsi_unsafe', 'nsi_phil']

# titles = ['Deterministic NSI', 'NSI w/ Uncertain DDFs',
#           'Probabilistic NSI', 'Prob. NSI w/ Philly Distributions']

# comp_plot['phil_loss'] = comp_plot[dam_col + '_phil']/1e6
# comp_plot['phil_rank'] = comp_plot['phil_loss'].rank(ascending=False)

# fig, ax = plt.subplots(figsize=(8, 8),
#                        sharex=True,
#                        sharey=True,
#                        nrows=2,
#                        ncols=2,
#                        dpi=300)

# for k, exp in enumerate(exps):
#     i = k//2
#     j = k%2

#     comp_plot['exp_loss'] = comp_plot[dam_col + '_' + exp]/1e6
#     comp_plot['exp_rank'] = comp_plot['exp_loss'].rank(ascending=False)
#     comp_plot['dam_bias'] = comp_plot['exp_loss'] - comp_plot['phil_loss']
#     comp_plot['rank_bias'] = comp_plot['exp_rank'] - comp_plot['phil_rank'] 

#     comp_plot['dam_pct_bias'] = 100*comp_plot['dam_bias']/comp_plot['phil_loss']+.01

#     total_exp_loss = comp_plot['exp_loss'].sum()
#     total_phil_loss = comp_plot['phil_loss'].sum()
#     total_pct_dev = np.round(100*(total_exp_loss-total_phil_loss)/total_phil_loss)

#     sns.scatterplot(
#     ax=ax[i, j],
#     y='exp_rank',
#     x='phil_rank',
#     edgecolor='gray',
#     hue='dam_pct_bias',
#     palette='bwr',
#     hue_norm=colors.TwoSlopeNorm(vmin=-60,
#                                  vcenter=0,
#                                  vmax=100),
#     data=comp_plot[(comp_plot['exp_rank'] <= 20) | (comp_plot['phil_rank'] <= 20)],
#     legend=False
#     )

#     ax[i, j].set_title(titles[k], fontsize=14)

#     ax[i, j].axline([0, 0], [1, 1], color='black', linestyle='--', alpha=0.5, zorder=0)

#     ax[i, j].set_xlabel('Damage Rank with Philly Structures', fontsize=12)
#     ax[i, j].set_ylabel('Damage Rank with\nLess Accurate Inventory', fontsize=12)
#     ax[i, j].tick_params(labelsize=12)

#     ax[i, j].annotate('Top 20th% Type 1 Error',
#                 (45, 1.5),
#                 xycoords='data',
#                 ha='center',
#                 color='red',
#                 size=12
#                 )

#     ax[i, j].annotate('Top\n20th%\nType 2\nError',
#                 (8, 42),
#                 xycoords='data',
#                 ha='center',
#                 color='blue',
#                 size=12
#                 )
    

#     ax[i, j].set_xlim([0, 68])
#     ax[i, j].set_ylim([0, 65])

#     ax[i, j].axvline(20, ymin=20/65., color='blue', linestyle='--', alpha=1, zorder=0)
#     ax[i, j].axhline(20, xmax=20/68., color='blue', linestyle='--', alpha=1, zorder=0)
#     ax[i, j].axvline(20, ymax=20/65., color='red', linestyle='--', alpha=1, zorder=0)
#     ax[i, j].axhline(20, xmin=20/68., color='red', linestyle='--', alpha=1, zorder=0)

# fig.tight_layout()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import mean_squared_error
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
from matplotlib.patches import Patch

def calculate_metrics(comp_df, exps, dam_col, baseline='phil'):
    """Calculate various skill metrics for each experiment compared to baseline."""
    metrics = {}
    
    # Baseline values
    baseline_loss = comp_df[f"{dam_col}_{baseline}"]/1e6
    baseline_val = comp_df[f"median_val_{baseline}"]
    baseline_rank = baseline_loss.rank(ascending=False)
    baseline_total = baseline_loss.sum()
    
    # Top 20% tracts in baseline
    top20_threshold = int(len(comp_df) * 0.1)
    top20_tracts = baseline_loss.nlargest(top20_threshold).index
    
    for exp in exps:
        exp_metrics = {}
        
        # Calculate losses and ranks
        exp_loss = comp_df[f"{dam_col}_{exp}"]/1e6
        exp_val = comp_df[f"median_val_{exp}"]
        exp_rank = exp_loss.rank(ascending=False)
        exp_total = exp_loss.sum()
        
        # Total discrepancy metrics
        exp_metrics['total_discrepancy_dollar'] = exp_total - baseline_total
        exp_metrics['total_discrepancy_pct'] = 100 * (exp_total - baseline_total) / baseline_total
        
        # RMSE metrics
        exp_metrics['rmse_tract'] = np.sqrt(mean_squared_error(baseline_loss, exp_loss))
        
        # Correlation between structure value and damages

        exp_metrics['corr_val_dam'] = (stats.pearsonr(exp_loss, exp_val)[0] -
                                       stats.pearsonr(baseline_loss, baseline_val)[0])
        

        # Rank correlation metrics
        exp_metrics['rank_correlation'] = stats.spearmanr(exp_rank, baseline_rank)[0]
        
        # Top percentage rank correlation
        exp_metrics['rank_correlation_top20'] = stats.spearmanr( 
            exp_rank[baseline_rank <= len(top20_tracts)],
            baseline_rank[baseline_rank <= len(top20_tracts)]
        )[0]
        
        # Type 1 and Type 2 errors for top 20%
        # Type 1: Baseline says it's top 20%, experiment says it's not
        type1_error = len(set(top20_tracts) - set(exp_loss.nlargest(top20_threshold).index))
        exp_metrics['type1_error_count'] = type1_error 
        exp_metrics['type1_pct'] = 100*type1_error/top20_threshold
        
        # Type 2: Experiment says it's top 20%, baseline says it's not
        type2_error = len(set(exp_loss.nlargest(top20_threshold).index) - set(top20_tracts))
        exp_metrics['type2_error_count'] = type2_error 
        exp_metrics['type2_pct'] = 100*type2_error/top20_threshold
        
        # Count of matched ranks
        exp_metrics['matched_top_rank_pct'] = (
             exp_rank[exp_rank <= len(top20_tracts)] -
             baseline_rank[baseline_rank <= len(top20_tracts)] == 0
            ).sum()*100/len(top20_tracts)


        # std error of ranks
        exp_metrics['std_rank'] = np.std(
            baseline_rank - exp_rank
        )

        metrics[exp] = exp_metrics
    
    return metrics

def plot_skill_metrics(metrics, exps, titles, colors, figsize=(10, 10)):
    """Create simplified small multiples plot of various skill metrics."""
    # Define the metrics to plot and their labels
    metrics_to_plot = [
        ('total_discrepancy_dollar', 'Total Discrepancy ($ Millions)'),
        ('total_discrepancy_pct', 'Total Discrepancy (%)'),
        ('rmse_tract', 'Tract RMSE ($ Millions)'),
        ('std_rank', 'Std Error of Tract Rank'),
        ('type1_pct', 'Misclassified as\nTop 10% Damaged Tracts'),
        ('matched_top_rank_pct', 'Correct Top 10% Damaged Rank'),
    ]
    
    # Set up the figure
    fig, axes = plt.subplots(2, 3,
                             figsize=figsize,
                             gridspec_kw={'wspace': .25,
                                          'hspace': .55},
                             dpi=300)
    axes = axes.flatten()
    
    # Set publication-quality style
    plt.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': ['Arial'],
        'font.size': 12
    })
    
    # Create each subplot
    for i, (metric_key, metric_label) in enumerate(metrics_to_plot):
        if i >= 8:  # Only plot 8 metrics
            break
            
        ax = axes[i]
        
        # Extract metric values for each experiment
        values = [metrics[exp][metric_key] for exp in exps]
        
        # Create bar plot with consistent colors by experiment
        x = np.arange(len(exps))
        bars = ax.bar(x, values, color=colors, width=0.7)
        
        # Add value labels on top of bars
        for bar, value in zip(bars, values):
            if isinstance(value, (int, float)):
                if 'pct' in metric_key:
                    value_text = f"{value:.0f}%"
                elif abs(value) >= 1000:
                    value_text = f"{value/1000:.1f}K"
                elif abs(value) >= 100:
                    value_text = f"{value:.0f}"
                elif value - int(value) == 0:
                    value_text = f"{value:.0f}"
                else:
                    value_text = f"{value:.2f}"
                
                # Position the text
                height = bar.get_height()
                if height < 0:
                    va = 'top'
                    height = height - 0.05 * max([abs(v) for v in values])
                else:
                    va = 'bottom'
                    height = height + 0.05 * max([abs(v) for v in values])
                
                ax.text(bar.get_x() + bar.get_width()/2., height,
                        value_text, ha='center', va=va, fontsize=10, 
                        color='black')
        
        # Set labels and title
        ax.set_title(metric_label, fontsize=12, pad=15)
        ax.set_xticks([])  # No x-labels on individual plots
        ax.tick_params(labelsize=12)
        
        # Add horizontal line at y=0 for metrics that can be negative
        if metric_key in ['total_discrepancy_dollar', 'total_discrepancy_percent']:
            ax.axhline(y=0, color='black', linestyle='-', alpha=0.5, linewidth=0.8)
        
        # Format y-axis for percentage metrics
        if 'pct' in metric_key:
            ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f'{y:.0f}%'))
            ax.set_ylim(bottom=0)  # Start at 0 for percentages
        
        # Set y-limits for correlation metrics
        if 'correlation' in metric_key:
            ax.set_ylim(0, 1.05)

        if 'dollar' in metric_key:
            ax.set_ylim(0, max([abs(v) for v in values]))
        
        # Add grid
        ax.grid(axis='y', linestyle='--', alpha=0.3, linewidth=0.5)
        
        # Remove top and right spines
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    
    # Hide the unused subplot
    # axes[len(axes)-1].axis('off')
    
    # Create legend patches
    legend_elements = []
    for i, title in enumerate(titles):
        legend_elements.append(Patch(facecolor=colors[i], label=title))
    
    # Add the legend to the empty subplot
    axes[len(axes)-1].legend(handles=legend_elements,
                             loc='center', 
                             frameon=True,
                             fancybox=False, 
                             ncol=2,
                             bbox_to_anchor=(-.8, -.5),
                             edgecolor='black')
    
    fig.tight_layout()
    
    return fig, axes

# Example usage
exps = ['nsi', 'nsi_ddfs', 'nsi_unsafe', 'nsi_phil', 'phil_nsi', 'phil_unsafe']
titles = ['Status Quo NSI Use', 'NSI w/ Uncertain DDFs Only',
          'NSI w/ All Uncertainty', 'NSI w/ All - Philly Structure Dist.',
          'Philly w/ All - NSI Structure Dist.',
          'Philly w/ All - Philly Structure Dist.']

# Define distinct colors for each method
colors = ['#EE7733', '#EE3377', '#CC3311', '#0077BB', '#33BBEE', '#009988']

# Calculate metrics
metrics = calculate_metrics(comp, exps, dam_col)

# Create the plot
fig, axes = plot_skill_metrics(metrics, exps, titles, colors, figsize=(10, 4))

In [ ]:
# Use the spatial links to add bfid to the nsi data
fd_bfid_lnk = dict(zip(lnk_nsi_loc['fd_id'], lnk_nsi_loc['bfid']))
nsi_match = nsi_inv_ens.reset_index()
nsi_match['bfid'] = nsi_match['fd_id'].map(fd_bfid_lnk)
# We will link up all the records across inventories
# including those w/o a match in the other in order to define
# different match categories
nsi_match.loc[nsi_match['bfid'].isnull(), 'bfid'] = nsi_match.loc[nsi_match['bfid'].isnull()]['fd_id']
match_df = nsi_match.merge(phil_inv_ens.reset_index(),
                           on='bfid',
                           suffixes=['_nsi', '_phil'],
                           how='outer')

matches = match_df.groupby(['bfid', 'num_story_phil', 'found_type_phil',
                            'occtype_phil', 'occtype_nsi',
                            'num_story_nsi', 'found_type_nsi']).size().reset_index()

matches['story_match'] = 0
matches.loc[matches['num_story_phil'] == matches['num_story_nsi'],
            'story_match'] = 1

matches['found_type_match'] = 0
matches.loc[matches['found_type_phil'] == matches['found_type_nsi'],
            'found_type_match'] = 1

matches['occ_match'] = 0
matches.loc[matches['occtype_phil'] == matches['occtype_nsi'],
            'occ_match'] = 1

matches.loc[(matches['story_match'] == 0) & (matches['found_type_match'] == 0),
            'Structure Matches'] = 'Location Only'

matches.loc[((matches['story_match'] == 1) &
             (matches['found_type_match'] == 1) &
             (matches['occ_match'] == 1)),
            'Structure Matches'] = 'All'

matches.loc[((matches['story_match'] == 1) &
             (matches['found_type_match'] == 1) &
             (matches['occ_match'] == 0)),
            'Structure Matches'] = 'Basement & Stories'

matches.loc[((matches['story_match'] == 1) &
             (matches['found_type_match'] == 0) &
             (matches['occ_match'] == 0)),
            'Structure Matches'] = 'Stories'

matches.loc[((matches['story_match'] == 0) &
             (matches['found_type_match'] == 1) &
             (matches['occ_match'] == 0)),
            'Structure Matches'] = 'Basement'

matches.loc[((matches['story_match'] == 0) &
             (matches['found_type_match'] == 0) &
             (matches['occ_match'] == 1)),
            'Structure Matches'] = 'Occupancy'

matches.loc[((matches['story_match'] == 1) &
             (matches['found_type_match'] == 0) &
             (matches['occ_match'] == 1)),
            'Structure Matches'] = 'Stories & Occupancy'

matches.loc[((matches['story_match'] == 0) &
             (matches['found_type_match'] == 1) &
             (matches['occ_match'] == 1)),
            'Structure Matches'] = 'Basement & Occuapncy'

match_agg_l = ['Stories', 'Basement', 'Occupancy', 'Stories & Occupancy',
               'Basement & Stories', 'Basement & Occuapncy']
matches.loc[matches['Structure Matches'].isin(match_agg_l),
            'Structure Matches'] = 'Location & Subset'

match_dict = dict(zip(matches['bfid'].astype(int), matches['Structure Matches']))

phil_only = match_df[(~match_df['bfid'].isin(matches['bfid'])) & (match_df['fd_id'].isnull())]
phil_only['Structure Matches'] = 'Unmatched Philly'
nsi_only = match_df[(~match_df['bfid'].isin(matches['bfid'])) & (match_df['fd_id'].notnull())]
nsi_only['Structure Matches'] = 'NSI Not In Philly'

phil_only_dict = dict(zip(phil_only['bfid'].astype(int), phil_only['Structure Matches']))
nsi_only_dict = dict(zip(nsi_only['bfid'].astype(int), nsi_only['Structure Matches']))

match_dict |= phil_only_dict
match_dict |= nsi_only_dict

In [ ]:
nsi_phil_id_dict = dict(zip(lnk_nsi_loc['fd_id'], lnk_nsi_loc['bfid']))
main_nsi = benchmarks['no_adj'].join(nsi_inv_ens).reset_index()

main_phil = all_results['original']['phil:no_adj']
# main_phil = main_phil.groupby('bfid')[dam_col].mean().reset_index()
main_phil = main_phil.merge(phil_inv_ens.reset_index(), on='bfid')

main_nsi['bfid'] = main_nsi['fd_id'].map(nsi_phil_id_dict)

# Calculate relative damage for each and include that in the mean step after merge
main_nsi['rel_loss'] = main_nsi[dam_col]/main_nsi['val_struct']
main_phil['rel_loss'] = main_phil[dam_col]/main_phil['val_s']

# Merge depths in
main_nsi[dg_id] = main_nsi['fd_id'].map(nsi_depths_df[dg_id])*3.28084
main_phil[dg_id] = main_phil['bfid'].map(phil_depths_df[dg_id])*3.28084

merge_cols = ['bfid', dam_col, dg_id, 'rel_loss', 'num_story', 'found_type', 'occtype']

# Do a quick update of occtype to RES1 if basement property
main_phil.loc[main_phil['found_type'] == 'B', 'occtype'] = 'RES1'

phil_mean = main_phil.groupby('bfid').agg({dam_col: 'mean',
                                           dg_id: 'first',
                                           'rel_loss': 'mean',
                                           'occtype': 'first',
                                           'num_story': 'first',
                                           'found_type': 'first'}).reset_index()

test = main_nsi.merge(phil_mean[merge_cols],
                      suffixes=['_nsi', '_phil'],
                      on='bfid',
                      how='outer')

# If bfid is null or Philly damage is 0, use fd_id in its place so we have a unique id
# We also look at Philly damage being 0 because if that's the case there's no depth
# for the property at that bfid and we want to use the fd_id instead for merging
# depths in later
fd_id_mask = test['bfid'].isnull() # | test[dam_col+'_phil'].isnull()
test.loc[fd_id_mask, 'bfid'] = test.loc[fd_id_mask, 'fd_id']

# NSI buildings can be stacked on top of each other so we'll aggregate these
# and treat them like one structure
test_gb = test.groupby(['bfid']).agg({dam_col + '_nsi': 'sum',
                                      dam_col + '_phil': 'first',
                                      dg_id + '_nsi': 'first',
                                      dg_id + '_phil': 'first',
                                      'rel_loss_nsi': 'first',
                                      'rel_loss_phil': 'first'}).reset_index().fillna(0)

test_gb['bfid'] = test_gb['bfid'].astype(int)

test_gb['nsi_rank'] = test_gb[dam_col+'_nsi'].rank(ascending=False, method='min')
test_gb['phil_rank'] = test_gb[dam_col+'_phil'].rank(ascending=False, method='min')

# Dam and rank diff
test_gb['diff'] = test_gb[dam_col+'_nsi'] - test_gb[dam_col+'_phil']
test_gb['diff_rel'] = test_gb['rel_loss_nsi'] - test_gb['rel_loss_phil']
test_gb['rank_diff'] = test_gb['nsi_rank'] - test_gb['phil_rank']
test_gb['diff_m'] = test_gb['diff']/1e6

# For visualization purposes, create an artificial high fillna value
test_gb.loc[test_gb['nsi_rank'].isnull(), 'nsi_rank'] = len(test_gb) 
test_gb.loc[test_gb['phil_rank'].isnull(), 'phil_rank'] = len(test_gb)


test_gb['Matches'] = test_gb['bfid'].map(match_dict)# .fillna('Philly Only (No Match)')

# Get depth back in 
test_gb['depth_ft'] = (test_gb['bfid'].map(phil_depths_df[dg_id])*3.28084)
test_gb.loc[test_gb['depth_ft'].isnull(),
            'depth_ft'] = test_gb['bfid'].map(nsi_depths_df[dg_id])*3.28084

# Id for "true" damage
test_gb['phil_dam'] = 0
test_gb.loc[test_gb['bfid'].isin(phil_mean['bfid']), 'phil_dam'] = 1

# Assign depths based on damage source
test_gb.loc[test_gb['phil_dam'] == 1, 'depth_ft'] = test_gb.loc[test_gb['phil_dam'] == 1, dg_id+'_phil']
test_gb.loc[test_gb['phil_dam'] == 0, 'depth_ft'] = test_gb.loc[test_gb['phil_dam'] == 0, dg_id+'_nsi']

# Update match column where no Philly damages
# test_gb.loc[test_gb['phil_dam'] == 0, 'Matches'] = 'NSI Only (No Match)'

# We only want to keep columns where there is damage in either record 
test_gb = test_gb.loc[(test_gb['phil_dam'] == 1) | (test_gb[dam_col+'_nsi'] > 0)]

# Get tract_id back in
test_gb['tract_id'] = test_gb['bfid'].map(phil_refs.set_index('bfid')['tract_id'])
test_gb.loc[test_gb['tract_id'].isnull(),
            'tract_id'] = test_gb['bfid'].map(nsi_refs.set_index('fd_id')['tract_id'])

# For cumulative discrepancies
# We need to merge in the NSI dam estimates to the Phil counterparts
# and then take the cumulative differences for each SOW
# We want this by each match type and overall, then we'll aggregate
# across SOWs for the mean of each and that's what we'll plot
# We merge that into test_gb

# The reason I decided not to do it this way is that the cumulative discrepancy
# within each SOW is not really of interest. The thing to compare in this context
# is what your "best guess" damage is using the ensemble approach vs. NSI approach.
# This would be the mean damage across SOWs and then we'd compare that to what you 
# get from NSI. There is much more variability than that (which the next plot shows)
# but conditioned on using the "best guess" we see that there is a large
# damage discrepancy. 

test_gb['depth_plot'] = test_gb['depth_ft'].round(1)
test_gb['cm_diff'] = test_gb.sort_values('depth_plot').groupby('Matches')['diff'].transform('cumsum')/1e6

test_gb['cm_diff_agg'] = test_gb.sort_values('depth_plot')['diff'].transform('cumsum')/1e6

In [ ]:
phil_cm_cols = ['bfid', dam_col, 'sow_ind']
nsi_cm_cols = ['bfid', dam_col, 'fd_id',  'sow_ind']

# Need to merge nsi entries with the same bfid before merging!
# We'll take the sum of the damage for NSI properties
# linked to the same Philly property
ens_nsi = main_nsi.copy()
ens_nsi.loc[ens_nsi['bfid'].isnull(), 'bfid'] = ens_nsi.loc[ens_nsi['bfid'].isnull(), 'fd_id']
ens_nsi = ens_nsi.groupby(['bfid']).agg({'fd_id': 'first',
                                          dam_col: 'sum'}).reset_index()

ens_nsi = ens_nsi.loc[np.repeat(ens_nsi.index, main_phil['sow_ind'].max() + 1)].reset_index(drop=True)
sow_ind = np.arange(len(ens_nsi)) % (main_phil['sow_ind'].max() + 1)
ens_nsi = pd.concat([ens_nsi, pd.Series(sow_ind, name="sow_ind")], axis=1)

test_cm = main_phil[phil_cm_cols].merge(ens_nsi[nsi_cm_cols],
                                        suffixes=['_phil', '_nsi'],
                                        on=['bfid', 'sow_ind'],
                                        how='outer')

fd_id_mask = test_cm['bfid'].isnull()
test_cm.loc[fd_id_mask, 'bfid'] = test_cm.loc[fd_id_mask, 'fd_id']

test_cm['bfid'] = test_cm['bfid'].astype(int)

test_cm['depth_ft'] = (test_cm['bfid'].map(phil_depths_df[dg_id])*3.28084)
test_cm.loc[test_cm['depth_ft'].isnull(),
            'depth_ft'] = test_cm['fd_id'].map(nsi_depths_df[dg_id])*3.28084


test_cm['diff'] = test_cm[dam_col+'_nsi'].fillna(0) - test_cm[dam_col+'_phil'].fillna(0)

# Get tract_id back in
test_cm['tract_id'] = test_cm['bfid'].map(phil_refs.set_index('bfid')['tract_id'])
test_cm.loc[test_cm['tract_id'].isnull(),
            'tract_id'] = test_cm['bfid'].map(nsi_refs.set_index('fd_id')['tract_id'])

# We'll use the bfid/Matches from the test_gb df because
# this lets us know about the Phil & NSI only properties
# There might be some SOWs where Phil damage is 0 and
# it wouldn't count as unmatched but it is. That 
# 0 discrepancy will affect the cumulative discrepancy curve,
# which is what we're trying to account for by aggregating
# across SOWs
test_cm['Matches'] = test_cm['bfid'].map(match_dict)

test_cm['cm_diff'] = test_cm.sort_values('depth_ft').groupby(['Matches', 'sow_ind'])['diff'].transform('cumsum')/1e6
test_cm['cm_diff_agg'] = test_cm.sort_values('depth_ft').groupby(['sow_ind'])['diff'].transform('cumsum')/1e6

test_cm_agg = test_cm.groupby(['bfid', 'depth_ft'])['cm_diff_agg'].mean().reset_index()
test_cm_match = test_cm.groupby(['bfid', 'Matches', 'depth_ft'])['cm_diff'].mean().reset_index()

test_cm_agg['depth_plot'] = test_cm_agg['depth_ft'].round(1)
test_cm_match['depth_plot'] = test_cm_match['depth_ft'].round(1)

In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import matplotlib.ticker as mtick

# Define consistent order for better visual alignment
hue_order = ['Location Only',
             'Location & Subset',
             'All',
             'NSI Not In Philly',
             'Unmatched Philly Res']

# Use a proper categorical palette instead of viridis
palette = sns.color_palette("Set2", n_colors=len(hue_order))
palette_dict = dict(zip(hue_order, palette))

# Create figure with custom grid layout
fig = plt.figure(figsize=(10, 8), dpi=300)
gs = gridspec.GridSpec(3, 2, width_ratios=[3, 2], height_ratios=[2, 2, 1],
                      wspace=0.25, hspace=0.05)

# Left column - original 3×1 layout
ax1 = fig.add_subplot(gs[0, 0])  # Top left - relative difference
ax2 = fig.add_subplot(gs[1, 0])  # Middle left - absolute difference
ax3 = fig.add_subplot(gs[2, 0])  # Bottom left - counts

# Right column - cumulative plot
ax4 = fig.add_subplot(gs[:, 1])  # Right side - spans all rows

# First create the depth bins if not already done
test_gb['depth_bins'] = pd.cut(test_gb['depth_ft'],
                              bins=[0, 1, test_gb['depth_ft'].max()])

# Panel 1: Relative difference boxplot
sns.boxplot(data=test_gb,
           x='depth_bins',
           y='diff_rel',
           hue='Matches',
           hue_order=hue_order,
           palette=palette_dict,
           showfliers=False,
           showmeans=True,
           meanprops={'markerfacecolor': 'firebrick', 'markeredgecolor': 'black', 'marker': 'D'},
           ax=ax1)

ax1.axhline(0, color='black', ls='--', alpha=.75)
ax1.set_xlabel('')  # Remove x-label as it's shared
ax1.set_ylabel('NSI - Mean Philly\n % Damage', size=14)
# Remove x-tick labels for top row
ax1.set_xticklabels([])
ax1.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))

# Panel 2: Absolute difference boxplot
sns.boxplot(data=test_gb,
           x='depth_bins',
           y='diff',
           hue='Matches',
           hue_order=hue_order,
           palette=palette_dict,
           showfliers=False,
           showmeans=True,
           meanprops={'markerfacecolor': 'firebrick', 'markeredgecolor': 'black', 'marker': 'D'},
           legend=False,
           ax=ax2)

ax2.axhline(0, color='black', ls='--', alpha=.75)
ax2.set_xlabel('')  # Remove x-label as it's shared
ax2.set_ylabel('NSI - Mean Philly\n $ Damage', size=14)
# Remove x-tick labels for middle row
ax2.set_xticklabels([])

# Panel 3: Count plot
sns.countplot(data=test_gb,
             x='depth_bins',
             hue='Matches',
             hue_order=hue_order,
             palette=palette_dict,
             legend=False,
             ax=ax3)

ax3.set_yscale('log')
ax3.set_xlabel('Depth Relative to Grade (Ft.)', size=14)
ax3.set_ylabel('Number of\nObservations', size=14)

# Panel 4: Cumulative discrepancy plot (right side)
# Let seaborn handle the categories directly
sns.lineplot(data=test_gb,
            x='depth_plot',
            y='cm_diff',
            hue='Matches',
            hue_order=hue_order,
            palette=palette_dict,
            ax=ax4,
            alpha=0.8,
            linewidth=2)

# Add the overall discrepancy line with enhanced styling
sns.lineplot(data=test_gb,
            x='depth_plot',
            y='cm_diff_agg',
            color='black',
            lw=3,
            label='Overall Discrepancy',
            ax=ax4)

# Improve the cumulative plot styling
ax4.axhline(0, color='gray', ls='--', alpha=.75, linewidth=1.5)
ax4.set_ylabel('Cumulative Damage Discrepancy ($ M)', size=14)
ax4.set_xlabel('Depth (Ft.)', size=14)
ax4.axvline(1, color='black', linestyle='--', alpha=.3)

# Add a shaded region around the zero line to emphasize the crossing points
ax4.axhspan(-0.5, 0.5, color='gray', alpha=0.1)

# Align y-labels
fig.align_ylabels([ax1, ax2, ax3])

# Optimize legend placement - centered at bottom
handles, labels = ax1.get_legend_handles_labels()
ax1.legend_.remove()  # Remove the original legend
legend = fig.legend(handles, labels,
                   title='Location and Characteristic Matches',
                   title_fontsize='x-large',
                   fontsize='x-large',  # Increased from 'medium' to 'large'
                   loc='upper center',
                   bbox_to_anchor=(0.5, 0.07),  # Position at bottom center
                   ncol=3)  # Spread horizontally to save space

# Remove the separate legend for the overall discrepancy line
ax4.legend_.remove()  # Remove the original legend

# Add the annotation label
ax4.annotate('Overall Discrepancy', 
            xy=(3.5, 60),  # Point to annotate
            fontsize=14)

# Adjust tick sizes
for ax in [ax1, ax2, ax3, ax4]:
    ax.tick_params(labelsize=12)

plt.tight_layout()
# Adjust the bottom margin to make room for the legend
plt.subplots_adjust(bottom=0.15)

In [ ]:
l_and_s_min = test_gb[test_gb['Matches'] == 'Location & Subset']['cm_diff'].min()
test_gb[(test_gb['Matches'] == 'Location & Subset') & (test_gb['cm_diff'] == l_and_s_min)]

In [ ]:
test_gb[test_gb['Matches'] == 'Location & Subset'].sort_values('depth_ft', ascending=False).iloc[0]

In [ ]:
test_gb[test_gb['Matches'] == 'All'].sort_values('depth_ft', ascending=False).iloc[0]

In [ ]:
test_gb[test_gb['Matches'] == 'Location Only'].sort_values('depth_ft', ascending=False).iloc[0]

In [ ]:
print(test_gb[test_gb['Matches'] == 'NSI Only (No Match)']['diff'].sum()/1e6)
print(len(test_gb[test_gb['Matches'] == 'NSI Only (No Match)']))
print(len(test_gb[test_gb['Matches'] == 'Philly Only (No Match)']))
print(test_gb[test_gb['Matches'] == 'Philly Only (No Match)']['diff'].sum()/1e6)

In [ ]:
non_nsi_only_depth_max = test_gb[test_gb['Matches'] != 'NSI Only (No Match)']['depth_ft'].max()
print(non_nsi_only_depth_max)
print(test_gb[test_gb['depth_ft'] > non_nsi_only_depth_max]['diff'].sum()/test_gb[test_gb['Matches'] == 'NSI Only (No Match)']['diff'].sum())
print(test_gb[test_gb['depth_ft'] > non_nsi_only_depth_max]['diff'].sum()/1e6)
print(len(test_gb[test_gb['depth_ft'] > non_nsi_only_depth_max]['diff']))

In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import matplotlib.ticker as mtick

# Define consistent order for better visual alignment
hue_order = ['Location Only',
             'Location & Subset',
             'All',
             'NSI Not In Philly',
             'Unmatched Philly']

# Define a colormap that groups matches and non matches together thematically
palette_dict = {'Location Only': '#0077BB',
                'Location & Subset': '#33BBEE',
                'All': '#009988',
                'NSI Not In Philly': '#CC3311',
                'Unmatched Philly': '#EE3377'}

# Create figure with custom grid layout
fig = plt.figure(figsize=(12, 6), dpi=300)
gs = gridspec.GridSpec(2, 2, height_ratios=[2, 1],
                       width_ratios=[3, 1],
                       wspace=.25,
                       hspace=0.1)

# Left column - original 3×1 layout
ax1 = fig.add_subplot(gs[0, 0])  # Top - absolute difference
ax2 = fig.add_subplot(gs[1, 0])  # Bottom - counts
ax4 = fig.add_subplot(gs[:, 1])

# First create the depth bins if not already done
test_gb['depth_bins'] = pd.cut(test_gb['depth_ft'],
                              bins=[0, 1, 2, 4, test_gb['depth_ft'].max()])
test_gb['diff_thou'] = test_gb['diff']/1e3

# Top: Absolute difference boxplot
sns.boxplot(data=test_gb,
           x='depth_bins',
           y='diff',
           hue='Matches',
           hue_order=hue_order,
           palette=palette_dict,
           showfliers=False,
           showmeans=True,
           meanprops={'markerfacecolor': 'firebrick',
                      'markeredgecolor': 'black', 'marker': 'D'},
           legend=True,
           ax=ax1)

ax1.axhline(0, color='black', ls='--', alpha=.75)
ax1.set_xlabel('')  # Remove x-label as it's shared
ax1.set_ylabel('NSI - Mean Philly\n $ Damage', size=14)
ax1.set_yscale('symlog')
ax1.set_ylim([-10e5, 10e5])
ax1.set_yticks([-1e6, -1e4, -1e2, 0, 1e2, 1e4, 1e6])
# Remove x-tick labels for middle row
ax1.set_xticklabels([])

# Bottom: Count plot
sns.countplot(data=test_gb,
             x='depth_bins',
             hue='Matches',
             hue_order=hue_order,
             palette=palette_dict,
             legend=False,
             ax=ax2)

ax2.set_yscale('log')
ax2.minorticks_off()
ax2.set_xlabel('Depth Relative to Grade (Ft.)', size=14)
ax2.set_ylabel('Number of\nObservations', size=14)


# Align y-labels
fig.align_ylabels([ax1, ax2])

# Panel 4: Cumulative discrepancy plot (right side)
# Let seaborn handle the categories directly
sns.lineplot(data=test_cm_match,
            x='depth_plot',
            y='cm_diff',
            hue='Matches',
            hue_order=hue_order,
            palette=palette_dict,
            ax=ax4,
            alpha=0.8,
            legend=False,
            linewidth=2)

# Add the overall discrepancy line with enhanced styling
sns.lineplot(data=test_cm_agg,
            x='depth_plot',
            y='cm_diff_agg',
            color='black',
            lw=5,
            label='Overall Discrepancy',
            legend=False,
            ax=ax4)

# Improve the cumulative plot styling
ax4.axhline(0, color='gray', ls='--', alpha=.75, linewidth=1.5)
ax4.set_ylabel('Mean Cumulative Discrepancy ($ M)', size=14)
ax4.set_xlabel('Depth (Ft.)', size=14)
# ax4.axvline(1, color='black', linestyle='--', alpha=.3)

# Add a shaded region around the zero line to emphasize the crossing points
ax4.axhspan(-0.5, 0.5, color='gray', alpha=0.1)

# Optimize legend placement - centered at bottom
# handles, labels = ax1.get_legend_handles_labels()
ax1.legend_.remove()  # Remove the original legend

legend_elements = [Patch(facecolor='none', edgecolor='none',
                         label='Match Types'),
                   Patch(facecolor=palette_dict['Location Only'],
                         label='Location Only'),
                   Patch(facecolor=palette_dict['Location & Subset'],
                         label='Location & Subset'),
                   Patch(facecolor=palette_dict['All'],
                         label='All'),
                   Patch(facecolor='none', edgecolor='none',
                         label='No Match Types'),
                   Patch(facecolor=palette_dict['NSI Not In Philly'],
                         label='NSI Not In Philly'),
                   Patch(facecolor=palette_dict['Unmatched Philly'],
                         label='Unmatched Philly'),]

legend = fig.legend(handles=legend_elements,# handles, labels,
                    # title='Location and Characteristic Matches',
                    title_fontsize='x-large',
                    fontsize='x-large', 
                    loc='upper center',
                    bbox_to_anchor=(0.5, -0.01), 
                    ncol=2) 


# Add the annotation label
ax4.annotate('Aggregate', 
            xy=(8, 52),  # Point to annotate
            fontsize=14)

# Adjust tick sizes
for ax in [ax1, ax2, ax4]:
    ax.tick_params(labelsize=12)


In [ ]:
nsi_phil_id_dict = dict(zip(lnk_nsi_loc['fd_id'], lnk_nsi_loc['bfid']))
nsi_run = all_results['original']['nsi_ddfs:no_adj'].set_index('fd_id')
main_nsi = nsi_run.join(nsi_inv_ens).reset_index()

main_phil = all_results['original']['phil:no_adj']
# main_phil = main_phil.groupby('bfid')[dam_col].mean().reset_index()
main_phil = main_phil.merge(phil_inv_ens.reset_index(), on='bfid')

main_nsi['bfid'] = main_nsi['fd_id'].map(nsi_phil_id_dict)
main_nsi.loc[main_nsi['bfid'].isnull(), 'bfid'] = main_nsi.loc[main_nsi['bfid'].isnull(), 'fd_id']

# Calculate relative damage for each and include that in the mean step after merge
main_nsi['rel_loss'] = main_nsi[dam_col]/main_nsi['val_struct']
main_phil['rel_loss'] = main_phil[dam_col]/main_phil['val_s']

# Merge depths in
main_nsi[dg_id] = main_nsi['fd_id'].map(nsi_depths_df[dg_id])*3.28084
main_phil[dg_id] = main_phil['bfid'].map(phil_depths_df[dg_id])*3.28084

merge_cols = ['bfid', dam_col, dg_id, 'rel_loss', 'num_story', 'found_type', 'occtype']

# Do a quick update of occtype to RES1 if basement property
main_phil.loc[main_phil['found_type'] == 'B', 'occtype'] = 'RES1'

phil_mean = main_phil.groupby('bfid').agg({dam_col: 'mean',
                                           dg_id: 'first',
                                           'rel_loss': 'mean',
                                           'occtype': 'first',
                                           'num_story': 'first',
                                           'found_type': 'first'}).reset_index()

nsi_mean = main_nsi.groupby('bfid').agg({dam_col: 'mean',
                                         dg_id: 'first',
                                         'fd_id': 'first',
                                         'rel_loss': 'mean',
                                         'occtype': 'first',
                                         'num_story': 'first',
                                         'found_type': 'first'}).reset_index()

test = nsi_mean.merge(phil_mean[merge_cols],
                      suffixes=['_nsi', '_phil'],
                      on='bfid',
                      how='outer')

# If bfid is null or Philly damage is 0, use fd_id in its place so we have a unique id
# We also look at Philly damage being 0 because if that's the case there's no depth
# for the property at that bfid and we want to use the fd_id instead for merging
# depths in later
fd_id_mask = test['bfid'].isnull() # | test[dam_col+'_phil'].isnull()
test.loc[fd_id_mask, 'bfid'] = test.loc[fd_id_mask, 'fd_id']

# NSI buildings can be stacked on top of each other so we'll aggregate these
# and treat them like one structure
test_gb2 = test.groupby(['bfid']).agg({dam_col + '_nsi': 'sum',
                                      dam_col + '_phil': 'first',
                                      dg_id + '_nsi': 'first',
                                      dg_id + '_phil': 'first',
                                      'rel_loss_nsi': 'first',
                                      'rel_loss_phil': 'first'}).reset_index().fillna(0)

test_gb2['bfid'] = test_gb2['bfid'].astype(int)

test_gb2['nsi_rank'] = test_gb2[dam_col+'_nsi'].rank(ascending=False, method='min')
test_gb2['phil_rank'] = test_gb2[dam_col+'_phil'].rank(ascending=False, method='min')

# Dam and rank diff
test_gb2['diff'] = test_gb2[dam_col+'_nsi'] - test_gb2[dam_col+'_phil']
test_gb2['diff_rel'] = test_gb2['rel_loss_nsi'] - test_gb2['rel_loss_phil']
test_gb2['rank_diff'] = test_gb2['nsi_rank'] - test_gb2['phil_rank']
test_gb2['diff_m'] = test_gb2['diff']/1e6

# For visualization purposes, create an artificial high fillna value
test_gb2.loc[test_gb2['nsi_rank'].isnull(), 'nsi_rank'] = len(test_gb2) 
test_gb2.loc[test_gb2['phil_rank'].isnull(), 'phil_rank'] = len(test_gb2)


test_gb2['Matches'] = test_gb2['bfid'].map(match_dict)# .fillna('Philly Only (No Match)')

# Get depth back in 
test_gb2['depth_ft'] = (test_gb2['bfid'].map(phil_depths_df[dg_id])*3.28084)
test_gb2.loc[test_gb2['depth_ft'].isnull(),
            'depth_ft'] = test_gb2['bfid'].map(nsi_depths_df[dg_id])*3.28084

# Id for "true" damage
test_gb2['phil_dam'] = 0
test_gb2.loc[test_gb2['bfid'].isin(phil_mean['bfid']), 'phil_dam'] = 1

# Assign depths based on damage source
test_gb2.loc[test_gb2['phil_dam'] == 1, 'depth_ft'] = test_gb2.loc[test_gb2['phil_dam'] == 1, dg_id+'_phil']
test_gb2.loc[test_gb2['phil_dam'] == 0, 'depth_ft'] = test_gb2.loc[test_gb2['phil_dam'] == 0, dg_id+'_nsi']

# We only want to keep columns where there is damage in either record 
test_gb2 = test_gb2.loc[(test_gb2['phil_dam'] == 1) | (test_gb2[dam_col+'_nsi'] > 0)]

In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import matplotlib.ticker as mtick

# Define consistent order for better visual alignment
hue_order = ['Location Only',
             'Location & Subset',
             'All',
             'NSI Only (No Match)',
             'Philly Only (No Match)']

# Use a proper categorical palette instead of viridis
palette = sns.color_palette("Set2", n_colors=len(hue_order))
palette_dict = dict(zip(hue_order, palette))

# Create figure with custom grid layout
fig, ax = plt.subplots(figsize=(12, 4), dpi=300,
                       nrows=2, ncols=2,
                       sharey='row',
                       sharex=True,
                       gridspec_kw={'height_ratios': [2, 1],
                                    'hspace': .05,
                                    'wspace': .05})

# Left column 
ax1 = ax[0, 0]  # Top - absolute difference
ax2 = ax[1, 0]  # Bottom - counts
# Right column
ax3 = ax[0, 1]  # Top - absolute difference
ax4 = ax[1, 1]  # Bottom - counts

# First create the depth bins if not already done
test_gb['depth_bins'] = pd.cut(test_gb['depth_ft'],
                                bins=[0, 1, 2, 4, test_gb2['depth_ft'].max()])
test_gb2['depth_bins'] = pd.cut(test_gb2['depth_ft'],
                                bins=[0, 1, 2, 4, test_gb2['depth_ft'].max()])

# Top: Absolute difference boxplot
sns.boxplot(data=test_gb,
           x='depth_bins',
           y='diff',
           hue='Matches',
           hue_order=hue_order,
           palette=palette_dict,
           showfliers=False,
           showmeans=True,
           meanprops={'markerfacecolor': 'firebrick', 'markeredgecolor': 'black', 'marker': 'D'},
           legend=True,
           ax=ax1)

ax1.axhline(0, color='black', ls='--', alpha=.75)
ax1.set_xlabel('')  # Remove x-label as it's shared
ax1.set_ylabel('NSI - Mean Philly\n $ Damage', size=14)
# Remove x-tick labels for middle row
# ax1.set_xticklabels([])

sns.boxplot(data=test_gb2,
           x='depth_bins',
           y='diff',
           hue='Matches',
           hue_order=hue_order,
           palette=palette_dict,
           showfliers=False,
           showmeans=True,
           meanprops={'markerfacecolor': 'firebrick', 'markeredgecolor': 'black', 'marker': 'D'},
           legend=False,
           ax=ax3)

ax3.axhline(0, color='black', ls='--', alpha=.75)
ax3.set_xlabel('')  # Remove x-label as it's shared
# ax3.set_ylabel('NSI - Mean Philly\n $ Damage', size=14)
# Remove x-tick labels for middle row

# Bottom: Count plot
sns.countplot(data=test_gb,
             x='depth_bins',
             hue='Matches',
             hue_order=hue_order,
             palette=palette_dict,
             legend=False,
             ax=ax2)

# ax2.set_yscale('log')
ax2.set_xlabel('Depth Relative to Grade (Ft.)', size=14)
ax2.set_ylabel('Number of\nObservations', size=14)

sns.countplot(data=test_gb2,
             x='depth_bins',
             hue='Matches',
             hue_order=hue_order,
             palette=palette_dict,
             legend=False,
             ax=ax4)

ax4.set_yscale('log')
ax4.set_xlabel('Depth Relative to Grade (Ft.)', size=14)
ax4.set_ylabel('', size=14)
ax3.set_ylabel('', size=14)


# Align y-labels
fig.align_ylabels([ax1, ax2, ax3, ax4])

# Optimize legend placement - centered at bottom
handles, labels = ax1.get_legend_handles_labels()
ax1.legend_.remove()  # Remove the original legend
legend = fig.legend(handles, labels,
                   title='Location and Characteristic Matches',
                   title_fontsize='x-large',
                   fontsize='x-large',  # Increased from 'medium' to 'large'
                   loc='upper center',
                   bbox_to_anchor=(0.5, -0.01),  # Position at bottom center
                   ncol=3)  # Spread horizontally to save space


# Adjust tick sizes
for ax in [ax1, ax2, ax3, ax4]:
    ax.tick_params(labelsize=12)

ax1.set_title('Without DDF Uncertainty for NSI', size=14)
ax3.set_title('With DDF Uncertainty for NSI', size=14)


In [ ]:
# Ok finally let's show a few examples (3?) of 
# everything matching, corresponding to pink above
# and how wide the damage distribution is...
# We can show relative damage on one row and
# $ damage on the other
# Then show how across SOWs the distributions are wide
# as a function of increasing depth. So even though
# mean lines up, there are lots of SOWs where
# the discrepancy is large (and maybe multimodal
# depending on first-floor elevation?)

# I say we do it for a low, mid, high depth
# which I found through
# test_gb[(test_gb['Matches'] == 'All') &
#         (test_gb['diff'].abs() < 1000)].sort_values('naccs_loss_009_phil')

# bfids & depths
# 72631 - 0.032810
# 396 - 1.523448
# 42739 - 4.985303

In [ ]:
def plot_building_damage_comparison(phil_data, nsi_data, bfids, 
                                    depth_col='depth_ft',  # Column name for depth
                                    n_bins=50, # number of bins for histogram
                                    dam_cols=['loss', 'rel_loss'],
                                    dam_labels=['Damage ($ Thousands)', 'Damage (%)'],
                                    scale_factors=[1e3, 1],
                                    ens_comp=None):
    """
    Create a generalized comparison of building damages between Philadelphia and NSI data.
    
    Parameters:
    -----------
    phil_data : DataFrame
        Philadelphia building data with damage columns
    nsi_data : DataFrame
        NSI building data with damage columns
    bfids : list
        List of building IDs to analyze
    depth_col : str
        Name of the column containing depth information
    n_bins: int
        Number of bins for histogram
    dam_cols : list
        List of damage column names to compare
    dam_labels : list
        Labels for the damage columns
    scale_factors : list
        Scaling factors for the damage values
    ens_comp : DataFrame
        (Optional) Ensemble data to plot as well
    """
    
    # Create figure with a grid layout
    fig = plt.figure(figsize=(10, 6), dpi=300)
    
    # Define colors for consistency
    philly_color = sns.color_palette('Set1')[1]
    ens_color = sns.color_palette('Set2')[1]
    nsi_color = 'red'

    # Process each building
    for i, bfid in enumerate(bfids):
        # Filter data for this building
        phil_plot = phil_data[phil_data['bfid'] == bfid]
        nsi_plot = nsi_data[nsi_data['bfid'] == bfid]
        if ens_comp is not None:
            ens_plot = ens_comp[ens_comp['bfid'] == bfid]
        else:
            ens_plot = None
        
        # Get depth directly from the data
        depth = phil_plot[depth_col].iloc[0]  # Use Philadelphia data for depth
        
        # Process each damage column
        for j, (dam_col, dam_label, scale_factor) in enumerate(zip(dam_cols, dam_labels, scale_factors)):
            # Calculate subplot position
            subplot_idx = i * len(dam_cols) + j + 1
            
            # Create a subplot with 2 rows (boxplot on top, histogram below)
            ax = plt.subplot(len(bfids), len(dam_cols), subplot_idx)
            
            # Create a gridspec for this subplot to have boxplot on top and histogram below
            gs = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=ax.get_subplotspec(), 
                                                 height_ratios=[1, 3], hspace=0)
            
            # Create the boxplot axes (top) and histogram axes (bottom)
            ax_box = fig.add_subplot(gs[0])
            ax_hist = fig.add_subplot(gs[1], sharex=ax_box)
            
            # Hide the main axes
            ax.axis('off')
            ax_box.axis('off')
            
            # Prepare Philadelphia data with ensemble
            temp = (phil_plot.groupby(['sow_ind'])[[dam_col]].sum()/scale_factor).reset_index()
            if ens_plot is not None:
                temp2 = (ens_plot.groupby(['sow_ind'])[[dam_col]].sum()/scale_factor).reset_index()
            
            # Create histogram
            counts, bins, patches = ax_hist.hist(temp[dam_col], bins=n_bins, color=philly_color, alpha=.75)
            if ens_plot is not None:
                ax_hist.hist(temp2[dam_col], bins=n_bins, color=ens_color, alpha=.75)
            
            # Configure histogram
            ax_hist.grid(False)
            if i == len(bfids) - 1:  # Only add x-label to bottom row
                ax_hist.set_xlabel(dam_label, size=16)
            
            if j == 0 and i == 1:  # Only add y-label to first column
                ax_hist.set_ylabel('Number of Ensemble Members', size=16)
            
            # Fix y-axis for histogram
            # ax_hist.set_ylim(0, counts.max() * 1.1)  # Set appropriate y-limit
            
            # Create boxplot
            sns.boxplot(ax=ax_box,
                       data=temp,
                       color=philly_color,
                       x=dam_col,
                       showmeans=True,
                       meanprops={'markerfacecolor': 'firebrick',
                                 'markeredgecolor': 'black',
                                 'marker': 'D'})
            
            if ens_plot is not None:
                sns.boxplot(ax=ax_box,
                            data=temp2,
                            color=ens_color,
                            x=dam_col,
                            showmeans=True,
                            meanprops={'markerfacecolor': 'firebrick',
                                        'markeredgecolor': 'black',
                                        'marker': 'D'})
            
            # Add NSI reference line
            nsi_value = nsi_plot[dam_col].sum()/scale_factor
            ax_hist.axvline(nsi_value, color=nsi_color)
            ax_box.axvline(nsi_value, color=nsi_color)
            
            # Configure boxplot
            # ax_box.set_xticks([])  # Remove x-ticks
            ax_box.set_xlabel('')  # Remove x-label
            ax_box.set_yticks([])  # Remove y-ticks

            ax_hist.tick_params(labelsize=12)
            
            ax_hist.xaxis.set_major_locator(plt.MaxNLocator(5))

            # Process the x ticks for % damage
            if j == 1:
                ax_hist.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, symbol=None, decimals=0))

            # Add row title for depth
            if j == 0:
                # ax_box.set_title(f"Building with Shared Location, Characteristics, and Inundation Depth of {depth:.2f} ft", fontsize=14, loc='left')
                ax.annotate(f"Matched Structure With Inundation of {depth:.2f} ft",
                            xy=(1.1, 1.05),
                            horizontalalignment='center',
                            xycoords='axes fraction',
                            fontsize=16)
        
    # Optimize legend placement - centered at bottom
    # Create legend elements
    legend_elements = [
        Patch(facecolor=philly_color, label='Philly w/ DDF Uncertainty'),
        Line2D([0], [0], color=nsi_color, lw=2, label='NSI w/o DDF Uncertainty'),
        Line2D([0], [0], marker='D', markerfacecolor='firebrick',
                label='Ensemble Mean', ls='', markeredgecolor='black', markersize=8)
    ]
    
    # Adjust layout
    plt.tight_layout()
    plt.subplots_adjust(wspace=.22, hspace=None)

    plt.legend(handles=legend_elements,
                fontsize='x-large',  # Increased from 'medium' to 'large'
                loc='center',
                bbox_to_anchor=(-.1, -0.83),  # Position at bottom center
                frameon=True,
                ncol=3
    )

    return fig

# Example usage:
bfids = [38716, 55691, 36592]

ens_comp = all_results['original']['nsi_ddfs:no_adj']
ens_comp['bfid'] = ens_comp['fd_id'].map(fd_bfid_lnk)
ens_comp['rel_loss'] = ens_comp[dam_col]/ens_comp['val_s']

fig = plot_building_damage_comparison(
    phil_data=main_phil, 
    nsi_data=main_nsi, 
    bfids=bfids,
    depth_col=dg_id,
    dam_cols=[dam_col, 'rel_loss'],
    # ens_comp=ens_comp
)

In [ ]:
def plot_building_damage_comparison(phil_data, nsi_data, ens_comp, bfids,
                                   depth_col='depth_ft',
                                   n_bins=50,
                                   dam_cols=['rel_loss', 'val_s', 'loss'],
                                   dam_labels=['Damage (%)', 'Value ($ Thousands)', 'Damage ($ Thousands)'],
                                   scale_factors=[1, 1e3, 1e3],
                                   phil_label='Philly Ensemble',
                                   ens_label='Alternate Ensemble'):
    """
    Create a generalized comparison of building damages between Philadelphia and NSI data.
    
    Parameters:
    -----------
    phil_data : DataFrame
        Philadelphia building data with damage columns
    nsi_data : DataFrame
        NSI building data with damage columns
    ens_comp : DataFrame
        Second ensemble data to compare with Philadelphia
    bfids : list
        List of building IDs to analyze
    depth_col : str
        Name of the column containing depth information
    n_bins: int
        Number of bins for histogram
    dam_cols : list
        List of damage column names to compare
    dam_labels : list
        Labels for the damage columns
    scale_factors : list
        Scaling factors for the damage values
    phil_label : str
        Label for the Philadelphia ensemble
    ens_label : str
        Label for the second ensemble
    """
    # Create figure with a grid layout
    fig = plt.figure(figsize=(10, 8), dpi=300)
    
    # Define colors for consistency
    philly_color = sns.color_palette('Set1')[1]
    ens_color = sns.color_palette('Set2')[1]
    nsi_color = 'red'
    
    # Process each building
    for i, bfid in enumerate(bfids):
        # Filter data for this building
        phil_plot = phil_data[phil_data['bfid'] == bfid]
        nsi_plot = nsi_data[nsi_data['bfid'] == bfid]
        ens_plot = ens_comp[ens_comp['bfid'] == bfid]
        
        # Get depth directly from the data
        depth = phil_plot[depth_col].iloc[0]  # Use Philadelphia data for depth
        
        # Process each damage column
        for j, (dam_col, dam_label, scale_factor) in enumerate(zip(dam_cols, dam_labels, scale_factors)):
            # Calculate subplot position
            subplot_idx = i * len(dam_cols) + j + 1
            
            # Create a subplot with 2 rows (boxplot on top, histogram below)
            ax = plt.subplot(len(bfids), len(dam_cols), subplot_idx)
            
            # Create a gridspec for this subplot to have boxplot on top and histogram below
            gs = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=ax.get_subplotspec(),
                                                 height_ratios=[1, 3], hspace=0)
            
            # Create the boxplot axes (top) and histogram axes (bottom)
            ax_box = fig.add_subplot(gs[0])
            ax_hist = fig.add_subplot(gs[1], sharex=ax_box)
            
            # Hide the main axes
            ax.axis('off')
            
            # Prepare data from both ensembles
            phil_values = (phil_plot.groupby(['sow_ind'])[[dam_col]].sum()/scale_factor).reset_index()[dam_col]
            ens_values = (ens_plot.groupby(['sow_ind'])[[dam_col]].sum()/scale_factor).reset_index()[dam_col]
            
            # Create combined DataFrame for seaborn
            if dam_col != 'val_s':
                combined_data = pd.DataFrame({
                    'value': pd.concat([phil_values, ens_values]),
                    'source': [phil_label] * len(phil_values) + [ens_label] * len(ens_values)
                })
            else:
                combined_data = pd.DataFrame({
                    'value': phil_values,
                    'source': [phil_label] * len(phil_values)
                })
            
            # Create histograms using seaborn
            sns.histplot(data=combined_data, x='value', hue='source', 
                        bins=n_bins, alpha=0.75, ax=ax_hist,
                        palette={phil_label: philly_color, ens_label: ens_color},
                        legend=False,
                        element="step", fill=True, stat="count")
            
            # Create boxplots using seaborn
            sns.boxplot(data=combined_data, x='value', y='source', 
                       orient='h', ax=ax_box,
                       hue='source',
                       legend=False,
                       palette={phil_label: philly_color, ens_label: ens_color},
                       showmeans=True,
                       meanprops={'markerfacecolor': 'firebrick',
                                 'markeredgecolor': 'black',
                                 'marker': 'D'})
            
            # Remove y-axis labels but keep the ticks for visual separation
            ax_box.set_yticklabels([])
            ax_box.set_yticks([])
            ax_box.set_xlabel('')
            ax_box.axis('off')  # Hide the boxplot axes
            
            # Add NSI reference line
            nsi_value = nsi_plot[dam_col].sum()/scale_factor
            ax_hist.axvline(nsi_value, color=nsi_color, linestyle='-', linewidth=2)
            ax_box.axvline(nsi_value, color=nsi_color, linestyle='-', linewidth=2)
            
            # Configure histogram
            ax_hist.grid(False)
            if i == len(bfids) - 1:  # Only add x-label to bottom row
                ax_hist.set_xlabel(dam_label, size=16)
            else:
                ax_hist.set_xlabel('')
            
            if j == 0 and i == 1:  # Only add y-label to first column of first row
                ax_hist.set_ylabel('Number of Ensemble Members', size=16)
            else:
                ax_hist.set_ylabel('')
            
            ax_hist.tick_params(labelsize=12)
            ax_hist.xaxis.set_major_locator(plt.MaxNLocator(5))
            
            # Process the x ticks for % damage
            if j == 0:
                ax_hist.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, symbol=None, decimals=0))
            
            # Remove the automatically generated legend
            if ax_hist.get_legend():
                ax_hist.get_legend().remove()
            
            # Add row title for depth
            # if j == 0:
                # ax.annotate(f"Matched Structure With Inundation of {depth:.2f} ft",
                #            xy=(1.1, 1.05),
                #            horizontalalignment='center',
                #            xycoords='axes fraction',
                #            fontsize=16)
            if j == 1:
                
                ax.set_title(f"Matched Structure With Inundation of {depth:.2f} ft",
                             size=14)

    # Create legend elements
    legend_elements = [
        Patch(facecolor=philly_color, alpha=.75, label=phil_label),
        Patch(facecolor=ens_color, alpha=.75, label=ens_label),
        Line2D([0], [0], color=nsi_color, lw=2, label='NSI w/o DDF Uncertainty'),
        Line2D([0], [0], marker='D', markerfacecolor='firebrick',
              label='Ensemble Mean', ls='', markeredgecolor='black', markersize=8)
    ]
    
    # Adjust layout
    plt.tight_layout()
    plt.subplots_adjust(wspace=.22, hspace=None)
    
    # Add the legend
    plt.legend(handles=legend_elements,
              fontsize='x-large',
              loc='center',
              bbox_to_anchor=(-.8, -.75),
              frameon=True,
              ncol=2
    )
    
    return fig

# Example usage:
bfids = [38716, 55691, 36592]

ens_comp = all_results['original']['nsi_ddfs:no_adj']
ens_comp['bfid'] = ens_comp['fd_id'].map(fd_bfid_lnk)
ens_comp['rel_loss'] = ens_comp[dam_col]/ens_comp['val_s']
main_nsi['val_s'] = main_nsi['val_struct'].copy()

fig = plot_building_damage_comparison(
    phil_data=main_phil, 
    ens_comp=ens_comp,
    nsi_data=main_nsi, 
    bfids=bfids,
    depth_col=dg_id,
    dam_cols=['rel_loss', 'val_s', dam_col],
    dam_labels= ['Damage (%)', 'Value ($ Thousands)', 'Damage ($ Thousands)'],
    scale_factors=[1, 1000, 1000],
    ens_label='NSI w/ DDF Uncertainty Only'
)

In [ ]:
# Check these bfids out to see why we get the kind
# of discrepancies we get
bfids_unmatched = [6374, 54289, 16856]
fig = plot_building_damage_comparison(
    phil_data=main_phil, 
    ens_comp=ens_comp,
    nsi_data=main_nsi, 
    bfids=bfids_unmatched,
    depth_col=dg_id,
    dam_cols=['rel_loss', 'val_s', dam_col],
    dam_labels= ['Damage (%)', 'Value ($ Thousands)', 'Damage ($ Thousands)'],
    scale_factors=[1, 1000, 1000],
    ens_label='NSI w/ DDF Uncertainty Only'
)

In [ ]:
test_gb[test_gb['Matches'] == 'All']['diff'].describe().round()

In [ ]:
test_cm[(test_cm['naccs_loss_009_phil'].notnull()) &
        (test_cm['naccs_loss_009_nsi'].notnull()) &
        (test_cm['Matches'] == 'All') &
        (test_cm['depth_ft'] > 1) &
        (test_cm['diff'] < -20e3)].sort_values('diff')

In [ ]:
ens_comp[ens_comp['bfid'] == 38716]

In [ ]:
# test_gb['tract_id'] = np.where(test_gb['tract_id_phil'] == 0,
#                                test_gb['tract_id_nsi'],
#                                test_gb['tract_id_phil'])

# Census tract with all 2 or more story buildings
# and no basement. Also RES3, but treating that as
# more of a methods/discussion detail for the case study
tract_ref = gpd.read_file(join(REF_DIR_I, FIPS, 'tract.gpkg'))[['GEOID', 'geometry']]
t_id = '42101010300'

phil_geo = phil_inv_out[['bfid', 'geometry']]
phil_geo = phil_geo.merge(phil_inv_ens.reset_index(), on='bfid')
phil_geo['match'] = phil_geo['bfid'].map(match_dict).fillna('Philly Unmatched')
phil_temp = phil_geo[phil_geo['tract_id'] == t_id]

nsi_geo = nsi_clip_out[['fd_id', 'geometry']]
nsi_geo = nsi_geo.merge(nsi_inv_ens.reset_index(), on='fd_id')
nsi_geo['bfid'] = nsi_geo['fd_id'].map(fd_bfid_lnk)
nsi_geo['match'] = nsi_geo['bfid'].map(match_dict).fillna('NSI Not In Philly')
nsi_temp = nsi_geo[nsi_geo['tract_id'] == t_id]


In [ ]:
# For plotting the depths
dg_filename = HAZ_FILEN.replace('{ens_num}', dg_id)

# Get a xarray.DataArray of the depth grid
rift_filep = join(HAZ_DIR_UZ, dg_filename)
ens_dg = rio.open_rasterio(rift_filep, masked=True).rio.write_crs(HAZ_CRS,
                                                                  inplace=True)

# We can create a mask based on the tract of interest
tract_sub = tract_ref[tract_ref['GEOID'] == t_id].to_crs(HAZ_CRS)

clipped_dg = ens_dg.rio.clip(tract_sub.geometry.values,
                             drop=True,
                             invert=False)
# 

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
import matplotlib.ticker as mtick
import seaborn as sns

fig, ax = plt.subplots(figsize=(10, 6), dpi=300)

# Create a custom colormap with the zero_color for masked values
cmap_with_zero = plt.cm.get_cmap('Blues').copy()
cmap_with_zero.set_bad('gray')

# Create a mask for zero values
zero_mask = clipped_dg < 0.01
da_masked = clipped_dg[0].where(~zero_mask)

# Plot the masked data
fg = (da_masked*3.28404).plot(ax=ax, cmap=cmap_with_zero, alpha=.75,
                                 vmin=0, vmax=8)

# cx.add_basemap(ax,
#                attribution=False,
#                crs=phil_temp.crs)

phil_temp[phil_temp['match'] == 'All'].plot(ax=ax, edgecolor='purple', color='none')
phil_temp[phil_temp['match'] != 'All'].plot(ax=ax, edgecolor='orange', color='none')
phil_temp[phil_temp['match'] == 'Unmatched Philly'].plot(ax=ax, edgecolor='red', color='none')
nsi_temp[nsi_temp['match'] != 'NSI Not In Philly'].plot(ax=ax, color='black', markersize=3)
nsi_temp[nsi_temp['match'] == 'NSI Not In Philly'].plot(ax=ax, color='red', markersize=3)

ax.axis('off')
ax.set_title('')

# ax.set_xlim([-.009-7.522e1, -.0053-7.522e1])
# ax.set_ylim([39.96928, 39.97145])

fg.colorbar.ax.tick_params(labelsize=12)
fg.colorbar.set_label(label='Water Depth (Ft.)',
                      rotation=270,
                      labelpad=20,
                      size=14)

legend_elements = [Line2D([0], [0], marker='o', ls='',
                          color='red',
                          label='NSI not in Philly Res.',
                          markerfacecolor='red', 
                          markersize=5),
                    Line2D([0], [0], marker='o', ls='',
                          color='black',
                          label='NSI in Philly Res.',
                          markerfacecolor='black',
                          markersize=5),
                   Patch(facecolor='none', edgecolor='red',
                         label='Unmatched Philly Res.'),
                    Patch(facecolor='none', edgecolor='purple',
                         label='Characteristics All Match'),
                    Patch(facecolor='none', edgecolor='orange',
                         label='Characteristics Don\'t All Match')]

ax.legend(handles=legend_elements,
          loc='center',
          fontsize=12,
          bbox_to_anchor=(.6, -.2))

In [ ]:
# Find bfid with different structure match types from the plot above
# We want to focus on one with all the structures matching and take a few different depths

# Let's start with a low depth example and build up
# 19370 is a bfid with 0 NSI damage. 2SNB, RES1. 
# We can find a few more examples of RES1 2SNB with higher
# depths now and we can show ensemble clouds of points around the
# point estimate for each

# ts_matches = matches[(matches['occtype_phil'] == 'RES1') & 
#                      (matches['num_story_phil'] == 2) & 
#                      (matches['found_type_phil'] == 'S') & 
#                      (matches['Structure Matches'] == 'All')]['bfid']
# test_gb[test_gb['bfid'].isin(ts_matches)].sort_values('depth_ft').iloc[120:140]

# 19370 - .726
# 36603 - 1.524
# 36592 - 3.858
# 31554 - 5.595
# 34758 - 9.481234


# For mismatches, repeat (change depth plot value to find new bfids)
# ts_mismatch = matches[(matches['occtype_phil'] == 'RES3A') & 
#                       (matches['num_story_phil'] == 2) & 
#                       (matches['found_type_phil'] == 'S') & 
#                       (matches['num_story_nsi'] == 2) &
#                       (matches['found_type_nsi'] == 'S') &
#                       (matches['Structure Matches'] == 'Location & Subset')]['bfid']
# test_gb[(test_gb['bfid'].isin(ts_mismatch)) &
#         (test_gb['depth_plot'] == 0)].sort_values('diff_rel')

# 17485, 61488, 26469, 34501, 34797

In [ ]:
# Load DDFs for plotting
naccs_ddfs = pd.read_parquet(join(VULN_DIR_I, 'physical', 'naccs_ddfs.pqt'))

naccs_2snb = naccs_ddfs[naccs_ddfs['ddf_type'] == '2SNB_RES1']
# Add low/mid/high to naccs_plot
naccs_2snb[['low', 'mid', 'high']] = pd.DataFrame(naccs_2snb['params'].tolist(),
                                                    index=naccs_2snb.index)

naccs_3snb = naccs_ddfs[naccs_ddfs['ddf_type'] == '3SNB_RES3A']
# Add low/mid/high to naccs_plot
naccs_3snb[['low', 'mid', 'high']] = pd.DataFrame(naccs_3snb['params'].tolist(),
                                                    index=naccs_3snb.index)

In [ ]:
# Create figure with custom grid layout
fig = plt.figure(figsize=(10, 8), dpi=300)

# Create a 2-row grid with the top row for spatial plot and bottom row for damage functions
# The bottom row will be split into two equal columns
gs = gridspec.GridSpec(2, 2, height_ratios=[1, 1], width_ratios=[2, 1])

# Create the spatial plot in the top row spanning both columns
ax_spatial = fig.add_subplot(gs[:, 0])

# Create the two damage function plots in the bottom row
ax_match = fig.add_subplot(gs[0, 1])
ax_mismatch = fig.add_subplot(gs[1, 1])

# SPATIAL PLOT CODE
cmap_with_zero = plt.cm.get_cmap('Blues').copy()
cmap_with_zero.set_bad('none')

# Create a mask for zero values
zero_mask = clipped_dg[0] <= 0.01
da_masked = clipped_dg[0].where(~zero_mask)

# Plot the masked data
fg = (da_masked*3.28084).plot(ax=ax_spatial, cmap=cmap_with_zero, alpha=.75,
                              add_colorbar=False,
                              vmin=0, vmax=8)

phil_temp[phil_temp['match'] == 'All'].plot(ax=ax_spatial, lw=2, edgecolor='purple', color='none')
phil_temp[phil_temp['match'] != 'All'].plot(ax=ax_spatial, lw=2, edgecolor='orange', color='none')
phil_temp[phil_temp['match'] == 'Unmatched Philly'].plot(ax=ax_spatial, lw=2, edgecolor='red', color='none')
nsi_temp[nsi_temp['match'] != 'NSI Not In Philly'].plot(ax=ax_spatial, color='black', markersize=10)
nsi_temp[nsi_temp['match'] == 'NSI Not In Philly'].plot(ax=ax_spatial, color='red', markersize=10)

ax_spatial.axis('off')
# ax_spatial.set_title('Spatial Distribution of\nBuildings and Flood Depths', size=16, pad=15)
ax_spatial.set_title('')

ax_spatial.set_xlim([-.0087-7.522e1, -.00538-7.522e1])
ax_spatial.set_ylim([39.96928, 39.97135])

# cx.add_basemap(ax_spatial,
#                source=cx.providers.Esri.WorldImagery,
#                attribution=False,
#                crs=phil_temp.crs)

cbax = ax_spatial.inset_axes([0.2, -0.075, 0.7, 0.05])

fig.colorbar(fg,
             ax=ax_spatial,
             cax=cbax,
             orientation='horizontal',
             # shrink=.8,
             # anchor=(.8, .9),
             # pad=.05,
             # fraction=.05
             extend='max',
             label='Water Depth (Ft.)')
fg.colorbar.ax.tick_params(labelsize=12)
fg.colorbar.set_label(label='Water Depth (Ft.)',
                     size=14)

# Spatial plot legend
spatial_legend_elements = [
    Line2D([0], [0], marker='o', ls='',
          color='red',
          label='NSI Not In Philly Res.',
          markerfacecolor='red',
          markersize=5),
    Line2D([0], [0], marker='o', ls='',
          color='black',
          label='NSI In Philly Res.',
          markerfacecolor='black',
          markersize=5),
    Patch(facecolor='none', edgecolor='red',
         label='Unmatched Philly Res.'),
    Patch(facecolor='none', edgecolor='purple',
         label='Characteristics All Match'),
    Patch(facecolor='none', edgecolor='orange',
         label='Characteristics Don\'t All Match')
]

# Position the spatial legend at the bottom of the spatial plot
ax_spatial.legend(handles=spatial_legend_elements,
                 loc='center',
                 fontsize=14,
                 bbox_to_anchor=(0.55, -0.39),
                 ncol=1)

# DAMAGE FUNCTION PLOTS

# Define variables and data
bfids = [19370, 36592, 31554, 34758]
bfids_m = [61488, 34501, 34797]

# Plot for matched buildings
naccs_2snb.plot(x='depth_ft',
               y='mid',
               ax=ax_match,
               lw=1,
               ls='dashed',
               color='#0077BB')

for y_name in ['low', 'high']:
    naccs_2snb.plot(x='depth_ft',
                   y=y_name,
                   ax=ax_match,
                   lw=2,
                   ls='solid',
                   color='#0077BB')

for i, bfid in enumerate(bfids):
    nsi_ffe = main_nsi[main_nsi['bfid'] == bfid]['found_ht'].values[0]
    nsi_d = main_nsi[main_nsi['bfid'] == bfid][dg_id].values[0]
    nsi_rel = main_nsi[main_nsi['bfid'] == bfid]['rel_loss'].values[0]
    nsi_d_adj = nsi_d - nsi_ffe
    
    phil_bfid_plot = main_phil[main_phil['bfid'] == bfid]
    phil_bfid_plot['d_adj'] = phil_bfid_plot[dg_id] - phil_bfid_plot['ffe']
    
    sns.scatterplot(data=phil_bfid_plot,
                   x='d_adj',
                   y='rel_loss',
                   color='purple',
                   ax=ax_match,
                   alpha=.035,)
    ax_match.scatter(nsi_d_adj,
                    y=nsi_rel,
                    s=50,
                    marker='D',
                    color='purple',
                    zorder=3,
                    edgecolor='black')

# Plot for mismatched buildings
naccs_3snb.plot(x='depth_ft',
               y='mid',
               ax=ax_mismatch,
               lw=1,
               ls='dashed',
               color='#0077BB')

for y_name in ['low', 'high']:
    naccs_3snb.plot(x='depth_ft',
                   y=y_name,
                   ax=ax_mismatch,
                   lw=2,
                   ls='solid',
                   color='#0077BB')

for i, bfid in enumerate(bfids_m):
    nsi_ffe = main_nsi[main_nsi['bfid'] == bfid]['found_ht'].values[0]
    nsi_d = main_nsi[main_nsi['bfid'] == bfid][dg_id].values[0]
    nsi_rel = main_nsi[main_nsi['bfid'] == bfid]['rel_loss'].values[0]
    nsi_d_adj = nsi_d - nsi_ffe
    
    phil_bfid_plot = main_phil[main_phil['bfid'] == bfid]
    phil_bfid_plot['d_adj'] = phil_bfid_plot[dg_id] - phil_bfid_plot['ffe']
    
    sns.scatterplot(data=phil_bfid_plot,
                   x='d_adj',
                   y='rel_loss',
                   color='orange',
                   ax=ax_mismatch,
                   alpha=.035,)
    ax_mismatch.scatter(nsi_d_adj,
                       y=nsi_rel,
                       s=50,
                       marker='D',
                       color='orange',
                       zorder=3,
                       edgecolor='black')

# Configure both damage function plots
for ax, title in zip([ax_match, ax_mismatch], 
                     ['Examples Of Depth\nDamage Function Match', 
                      'Examples Of Depth\nDamage Function Mismatch']):
    ax.set_ylim([-.025, .7])
    xlow = naccs_2snb[naccs_2snb['high'] == 0]['depth_ft'].max() - .5
    xhigh = naccs_2snb['depth_ft'].max()
    
    # Remove auto-generated legends
    if ax.get_legend() is not None:
        ax.get_legend().remove()
    
    ax.set_xlim([xlow, xhigh])
    # ax.set_title(title, size=14)

ax_mismatch.set_xlabel('Depth Relative to First Floor (Ft.)', size=14)
ax_mismatch.tick_params('both', labelsize=12)
ax_match.tick_params('y', labelsize=12)
ax_match.set_xlabel('')
ax_match.tick_params('x', which='both', labelbottom=False)

# Only set y-label on the left plot
ax_match.set_ylabel('Percent Damage', size=14)
ax_mismatch.set_ylabel('Percent Damage', size=14)
ax_match.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, symbol='%', decimals=0))
ax_mismatch.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, symbol='%', decimals=0))

# Remove y-tick labels from right plot
# ax_mismatch.tick_params(axis='y', which='both', labelleft=False)

# Create legend for damage function plots
damage_legend_elements = [
    Line2D([0], [0], marker='D', ls='',
          color='black',
          label='NSI Point Estimate',
          markerfacecolor='black',
          markersize=5),
    Line2D([0], [0], marker='o', ls='',
          color='gray',
          alpha=.5,
          label='Sampled Damage Estimate',
          markerfacecolor='gray',
          markersize=5),
    Line2D([0], [0], marker='', ls='--',
          color='#0077BB',
          alpha=1,
          label='Damage Best Estimate'),
    Line2D([0], [0], marker='', ls='-',
          lw=2,
          color='#0077BB',
          alpha=1,
          label='Uncertainty Bounds'),
]

# Add the damage function legend between the two bottom plots
ax_mismatch.legend(handles=damage_legend_elements,
          loc='center',
          ncol=1,
          fontsize=14,
          bbox_to_anchor=(0.45, -0.51))

# Add annotations
ax_match.annotate('Examples of Single\nFamily Res. In Both',
                 xy=(-1.7, .59),
                 xycoords='data',
                 horizontalalignment='left',
                 verticalalignment='center',
                 size=14)

ax_mismatch.annotate('Examples of\nMultifamily Res. In\nPhilly Only',
                    xy=(-1.7, .59),
                    xycoords='data',
                    horizontalalignment='left',
                    verticalalignment='center',
                    size=14)

# Adjust layout
plt.tight_layout()
# Add extra space at the bottom for the legend
# plt.subplots_adjust(bottom=0.12)

In [ ]:
# If I do end up revisiting this, need to make sure we only
# process the tracts in comp_geo (or maybe there's something wrong with the
# way that was processed in the first place and we should just use test_gb..)

# match_props = (test_gb.groupby(['tract_id', 'Matches']).size()/test_gb.groupby(['tract_id']).size()).rename('prop').reset_index()
# d_props = (test_gb.groupby(['tract_id', 'depth_bins']).size()/test_gb.groupby(['tract_id']).size()).rename('prop').reset_index()

# # We only have two options for this so can keep one category
# d_props = d_props[d_props['depth_bins'] == pd.Interval(0, 1)]

# # Link each of these up to census tract geodataframes
# # When plotting the match proportions we can just loop through
# # each match type and subset on the type and plot proportions
# # Will be helpful to get a colormap that links the types
# # to the same colors as the plot with categories before
# # We can actually use the palette from before as long 
# # as we run that figure creation... Can also just
# # create it based on unique values
# # palette = sns.color_palette("Set2", n_colors=len(hue_order))
# # palette_dict = dict(zip(hue_order, palette))
# hue_order = ['Location Only',
#              'Location & Subset',
#              'All',
#              'NSI Only (No Match)',
#              'Philly Only (No Match)']
# palette = sns.color_palette("Set2", n_colors=len(hue_order))
# palette_dict = dict(zip(hue_order, palette))

# m_prop_geo = tract_ref.merge(match_props,
#                              how='inner',
#                              left_on='GEOID',
#                              right_on='tract_id')
# d_prop_geo = tract_ref.merge(d_props,
#                              how='inner',
#                              left_on='GEOID',
#                              right_on='tract_id')

In [ ]:
# import contextily as cx
# from matplotlib import ticker
# import matplotlib.colors as colors
# import matplotlib.gridspec as gridspec
# from mpl_toolkits.axes_grid1 import make_axes_locatable
# import seaborn as sns

# # Create figure
# fig = plt.figure(figsize=(8, 14), dpi=300)

# # Create two GridSpecs
# # Top GridSpec for the maps
# gs = gridspec.GridSpec(6, 2,
#                        hspace=.4,
#                        wspace=.05)

# # Create axes for maps
# ax0 = fig.add_subplot(gs[0:3, 1])
# ax1 = fig.add_subplot(gs[3:6, 1])

# # Create axes for scatterplot and boxplot
# # ax2 = fig.add_subplot(gs[0:2, 0])  # Depth proportions
# ax3 = fig.add_subplot(gs[0:2, 0])  # Location & Subset
# ax4 = fig.add_subplot(gs[2:4, 0])  # NSI Only
# ax5 = fig.add_subplot(gs[4:6, 0])  # Philly Only

# comp_plot = comp_geo.to_crs(tract_ref.crs)

# cmap = 'YlOrRd'

# comp_plot['nsi_loss'] = comp_plot[dam_col + '_nsi']/1e6
# comp_plot['phil_loss'] = comp_plot[dam_col + '_phil']/1e6

# comp_plot['dam_bias'] = comp_plot['nsi_loss'] - comp_plot['phil_loss']
# comp_plot['rank_bias'] = (comp_plot['nsi_loss'].rank(ascending=False) -
#                           comp_plot['phil_loss'].rank(ascending=False))

# total_nsi_loss = comp_plot['nsi_loss'].sum()
# total_phil_loss = comp_plot['phil_loss'].sum()

# vmin = min(comp_plot['nsi_loss'].min(), comp_plot['phil_loss'].min())
# vmax = max(comp_plot['nsi_loss'].max(), comp_plot['phil_loss'].max())
# vcenter=1

# comp_plot.plot(ax=ax0, column='phil_loss', cmap='Reds',
#                legend=True,
#                legend_kwds={'pad': .03,
#                            'shrink': .75},
#                vmin=0, vmax=comp_plot['phil_loss'].max())
# comp_plot.plot(ax=ax1, column='dam_bias', cmap='bwr',
#                legend=True,
#                legend_kwds={'pad': .03,
#                            'shrink': .75},
#                norm=colors.TwoSlopeNorm(vmin=comp_plot['dam_bias'].min(),
#                                         vcenter=0,
#                                         vmax=comp_plot['dam_bias'].max()))

# tract_ref.plot(ax=ax0, edgecolor='black', lw=.5, color='none')
# tract_ref.plot(ax=ax1, edgecolor='black', lw=.5, color='none')

# axes = [ax0, ax1]
# for i in range(len(axes)):
#     axes[i].tick_params(
#         axis="both",
#         which="both",
#         bottom=False,
#         left=False,
#         labelbottom=False,
#         labelleft=False,
#     )
#     axes[i].axis('off')

# # Add titles
# axes[0].set_title("Damages ($ M) w/\nPhilly Structures", fontsize=14)
# axes[1].set_title("Deviation ($ M) Introduced by\nNational Structure Inventory", fontsize=14)

# # Proportion small multiples
# d_prop_geo.plot(column='prop',
#                 cmap='Blues',
#                 ax=ax2)


# prop_axes = [ax3, ax4, ax5]
# hue_order_sub = ['Location & Subset',
#                 'NSI Only (No Match)',
#                 'Philly Only (No Match)']
# import matplotlib as mpl
# for i, hue in enumerate(hue_order_sub):
#     cmap_list = ['white', palette_dict[hue]]
#     custom_cmap = mpl.colors.LinearSegmentedColormap.from_list('custom', cmap_list)

#     m_prop_geo[m_prop_geo['Matches'] == hue].plot(column='prop',
#                                                     cmap=custom_cmap,
#                                                     ax=prop_axes[i])
#     prop_axes[i].set_title(hue)
#     prop_axes[i].axis('off')
#     tract_ref.plot(ax=prop_axes[i], edgecolor='black', lw=.5, color='none')

# # Update colorbar tick label size
# for ax in fig.axes:
#     if ax._axes.get_label() == '<colorbar>':
#         ax.tick_params(labelsize='14')

# ax2.axis('off')

In [ ]:
# That didn't work as expected. I want to try 
# doing the rank order discrepancy plot (with discrepancy color map for dots)
# when using different approaches for getting tract damage estimates
# For all ensemble based approaches, we will calculate damage by aggregating 
# across SOWs (add up damages for census tracts in each SOW then take average
# across all of them). The reason we don't do that for tract_gb is that
# we are explicitly looking at average discrepancy across structures
# The cumulative difference plot is misleading though. Could calculate
# the constituent parts across SOWs and then plot the average discrepancies
# across SOWs. Should probably do it this way. 